# Random forest variable-star classifier

Three independent stages: build an archive training sample, train a nine-feature classifier, and classify a prepared feature table. Each stage is disabled by default. Configure the paths and stage flags below before running it. See README.md for the feature conventions, catalogue sources and limitations.


## Setup

Paths are relative to the current notebook working directory. Supply a frozen training CSV and/or saved model to skip archive harvesting. No packages are installed by this notebook.


In [ ]:
from pathlib import Path
import importlib.util

PROJECT_DIR = Path.cwd().resolve()
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
CACHE_ROOT = PROJECT_DIR / "cache"
RAW_DIR = CACHE_ROOT / "catalogues"
FEATURE_CACHE_ROOT = CACHE_ROOT / "features"
CHECKPOINT_ROOT = CACHE_ROOT / "checkpoints"
LIGHTCURVE_CACHE_DIR = CACHE_ROOT / "lightcurves"

RUN_BUILD_TRAINING = False
RUN_TRAIN_MODEL = False
RUN_PREDICT = False

CANDIDATES_PER_CLASS = 2500
N_WORKERS = 1
OGLE_N_WORKERS = 1
RANDOM_STATE = 42
REQUIRE_ALL_TARGET_CLASSES_FOR_TRAINING = True
MIN_CONFIDENCE_TO_LABEL = 0.45
PREDICTION_CONFIDENCE_OVERRIDE = None

TRAINING_CSV = DATA_DIR / "variable_star_training_9features_complete.csv"
MODEL_PATH = OUTPUT_DIR / "variable_star_classifier_9features.joblib"
TRAINING_REPORT_CSV = OUTPUT_DIR / "variable_star_model_training_report_9features.csv"
TRUTH_TABLE_CSV = OUTPUT_DIR / "variable_star_training_truth_table_9features.csv"
PREDICTION_INPUT_CSV = DATA_DIR / "prediction_features.csv"
PREDICTION_OUTPUT_CSV = OUTPUT_DIR / "predictions.csv"

ML_FEATURES = ["logP", "amplitude", "MG", "BP_RP0", "skewness", "stetson_J",
               "delta_wesenheit_cepheid", "R21", "R31"]
TARGET_CLASSES = ["RRLYR", "DSCT", "GDOR", "CEP", "EB", "MIRA", "SRV", "ROT", "RS"]
ID_COLUMNS = ("source_id", "gaia_id", "tic_id", "ogle_id", "star_id", "candidate_key")

def require_packages(packages):
    missing = [pip_name for module, pip_name in packages.items()
               if importlib.util.find_spec(module) is None]
    if missing:
        raise ImportError("Install these packages in the notebook environment before running this stage: "
                          + ", ".join(missing))

def read_csv_with_ids(path, **kwargs):
    import pandas as pd
    path = Path(path).expanduser()
    header = pd.read_csv(path, nrows=0)
    dtype = {name: "string" for name in ID_COLUMNS if name in header.columns}
    return pd.read_csv(path, dtype=dtype, **kwargs)

def prepare_output_path(path):
    path = Path(path).expanduser()
    path.parent.mkdir(parents=True, exist_ok=True)
    return path

def sha256_file(path):
    import hashlib
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


## Stage 1: build the training table

This optional stage queries Gaia DR3, the Zhou catalogue, OGLE and MAST/TESS. It can take many hours. Completed candidates are checkpointed under cache/. The feature definitions and class-specific period searches retain the original workflow. Start with one worker; network/library timeouts do not guarantee that every remote request can be cancelled immediately.


In [ ]:
if RUN_BUILD_TRAINING:
    require_packages({"pandas":"pandas", "numpy":"numpy", "astroquery":"astroquery",
                      "astropy":"astropy", "tqdm":"tqdm", "scipy":"scipy",
                      "requests":"requests", "lightkurve":"lightkurve"})
    import io
    import csv
    import os
    import re
    import time
    import json
    import socket
    import threading
    import random
    from pathlib import Path
    import pandas as pd
    import numpy as np
    import requests
    from scipy import stats
    from tqdm.auto import tqdm
    from astropy.table import Table
    from astropy.timeseries import LombScargle
    from astropy.coordinates import SkyCoord
    import astropy.units as u
    from astroquery.gaia import Gaia
    from astroquery.utils.tap.core import TapPlus
    from astroquery.mast import Catalogs
    from astroquery.xmatch import XMatch
    import lightkurve as lk
    
    def make_progress(*args, **kwargs):
        return tqdm(*args, **kwargs)
    N_PER_CLASS = CANDIDATES_PER_CLASS
    MIN_CONFIDENCE = 0.7
    REQUEST_TIMEOUT = 60
    CATALOG_DOWNLOAD_TIMEOUT = 300
    SOCKET_TIMEOUT = 90
    GAIA_TAP_TIMEOUT = 240
    TRANSIENT_ABORT_AFTER = 30
    AUX_ABORT_AFTER = 8
    OGLE_HTTP_ATTEMPTS = 5
    OGLE_HTTP_BASE_SLEEP = 4.0
    TRANSIENT_PAUSE_SECONDS = 60
    TRANSIENT_MAX_PAUSES = 6
    GAIA_TOP_INITIAL = max(2000, CANDIDATES_PER_CLASS * 20)
    GAIA_TOP_ESCALATION = 4
    GAIA_TOP_MAX_ATTEMPTS = 3
    GAIA_BATCH_SIZE = 1000
    GAIA_PULSATOR_DETAIL_BLOCK = 500
    ZHOU_GAIA_BLOCK = 500
    EB_CROSSMATCH_BLOCK = 1000
    GAIA_CROSSMATCH_RADIUS_ARCSEC = 2.0
    TESS_AUTHOR_PRIORITY = ('SPOC', 'TESS-SPOC', 'QLP', 'GSFC-ELEANOR-LITE', 'TGLC', 'PATHOS', 'CDIPS', 'TASOC')
    TESS_HARVEST_MAX_G = 16.5
    MAX_POINTS_FOR_LOMB_SCARGLE = 6000
    MIN_LIGHTCURVE_POINTS = 20
    MAX_PARALLAX_ERROR_RATIO = 1.0
    CEPHEID_WESENHEIT_COLOR_COEFF = 1.9
    CEPHEID_PL_SLOPE = -3.853058319974742
    CEPHEID_PL_INTERCEPT = -1.0689761833751459
    CEPHEID_PL_SIGMA = 1.4891618950080006
    CLASS_PERIOD_RANGES = {'DSCT': (0.02, 0.35), 'GDOR': (0.2, 6.0), 'RS': (0.2, 27.0), 'ROT': (0.1, 30.0)}
    STRICT_REQUIRED_NUMERIC = ML_FEATURES + ['P', 'amplitude', 'A1', 'n_points', 'phot_g_mean_mag', 'bp_rp', 'parallax', 'parallax_error', 'ag_gspphot', 'ebpminrp_gspphot']
    STRICT_REQUIRED_NUMERIC = list(dict.fromkeys(STRICT_REQUIRED_NUMERIC))
    FINAL_COLUMNS = ['source_id', 'tic_id', 'ogle_id', 'ra', 'dec', 'star_type', 'label_source', 'confidence', 'phot_g_mean_mag', 'bp_rp', 'parallax', 'parallax_error', 'ag_gspphot', 'ebpminrp_gspphot', 'MG', 'BP_RP0', 'cepheid_wesenheit_g', 'delta_wesenheit_cepheid', 'skewness', 'stetson_J', 'P', 'logP', 'amplitude', 'A1', 'R21', 'phi21', 'R31', 'phi31', 'n_points', 'catalogue_used', 'feature_source', 'quality_flag']
    ID_STRING_COLUMNS = ['source_id', 'tic_id', 'ogle_id', 'candidate_key']
    GAIA_CLASS_MAP = {'RR': 'RRLYR', 'CEP': 'CEP'}
    GAIA_SPECIALIZED_TABLES = {'RR': 'gaiadr3.vari_rrlyrae', 'CEP': 'gaiadr3.vari_cepheid'}
    GAIA_SPEC_COLUMN_MAP = {'P': 'pf', 'amplitude': 'peak_to_peak_g', 'A1': 'fund_freq1_harmonic_ampl_g', 'R21': 'r21_g', 'phi21': 'phi21_g', 'R31': 'r31_g', 'phi31': 'phi31_g', 'n_points': 'num_clean_epochs_g'}
    ZHOU_URL = 'https://zenodo.org/records/13840760/files/GaiaDR3_vari_DSCTgDorSXPhe_Validated_R2.csv?download=1'
    ZHOU_RAW_PATH = RAW_DIR / 'zhou_raw.csv'
    OGLE_REGIONS = [('blg', 'https://www.astrouw.edu.pl/ogle/ogle4/OCVS/blg/ecl', 'phot_ogle4/I'), ('lmc', 'https://www.astrouw.edu.pl/ogle/ogle4/OCVS/lmc/ecl', 'phot/I'), ('smc', 'https://www.astrouw.edu.pl/ogle/ogle4/OCVS/smc/ecl', 'phot/I')]
    OGLE_REGION_PRIORITY = {'local': -1, 'blg': 0, 'lmc': 1, 'smc': 2}
    OGLE_CATALOG_CACHE = RAW_DIR / 'ogle_eb_raw.csv'
    OGLE_LPV_SOURCES = {'MIRA': ('blg', 'https://www.astrouw.edu.pl/ogle/ogle4/OCVS/blg/lpv', 'phot_ogle4/I', 'Miras.dat', 2, 3), 'SRV': ('blg', 'https://www.astrouw.edu.pl/ogle/ogle3/OIII-CVS/blg/lpv', 'phot/I', 'SRVs.dat', 4, 5)}
    OGLE_LPV_CATALOG_CACHE = {'MIRA': RAW_DIR / 'ogle_mira_raw.csv', 'SRV': RAW_DIR / 'ogle_srv_raw.csv'}
    GAIA_ROTATION_TABLE = 'gaiadr3.vari_rotation_modulation'
    GAIA_RS_CLASS_NAME = 'RS'
    GAIA_ROT_RAW_PATH = RAW_DIR / 'gaia_rot_raw.csv'
    GAIA_RS_RAW_PATH = RAW_DIR / 'gaia_rs_raw.csv'
    try:
        Gaia.ROW_LIMIT = -1
    except Exception:
        pass
    _rng = np.random.default_rng(RANDOM_STATE)
    _WARNINGS = []
    FEATURE_DEFINITION_VERSION = "legacy-nine-feature-v1"
    extraction_configuration = {
        "feature_definition_version": FEATURE_DEFINITION_VERSION,
        "features": ML_FEATURES, "period_ranges": CLASS_PERIOD_RANGES,
        "max_ls_points": MAX_POINTS_FOR_LOMB_SCARGLE,
        "min_lc_points": MIN_LIGHTCURVE_POINTS,
        "tess_author_priority": TESS_AUTHOR_PRIORITY,
        "tess_max_g": TESS_HARVEST_MAX_G,
        "max_parallax_error_ratio": MAX_PARALLAX_ERROR_RATIO,
        "cepheid_wesenheit": [CEPHEID_WESENHEIT_COLOR_COEFF, CEPHEID_PL_SLOPE,
                               CEPHEID_PL_INTERCEPT, CEPHEID_PL_SIGMA],
        "min_catalogue_confidence": MIN_CONFIDENCE,
        "random_state": RANDOM_STATE,
    }
    import hashlib
    extraction_id = hashlib.sha256(json.dumps(extraction_configuration, sort_keys=True).encode()).hexdigest()[:16]
    CACHE_DIR = FEATURE_CACHE_ROOT / extraction_id
    CHECKPOINT_DIR = CHECKPOINT_ROOT / extraction_id
    for directory in (RAW_DIR, CACHE_DIR, CHECKPOINT_DIR, OUTPUT_DIR, TRAINING_CSV.parent):
        directory.mkdir(parents=True, exist_ok=True)
    (CHECKPOINT_DIR / "extraction_configuration.json").write_text(json.dumps(extraction_configuration, indent=2))
    
    _GAIA_MIRROR = TapPlus(url='https://gaia.ari.uni-heidelberg.de/tap')
    _GAIA_TAP_SERVICES = [('ARI mirror', _GAIA_MIRROR), ('ESA', Gaia)]
    
    def refresh_gaia_tap_services():
        global _GAIA_MIRROR, _GAIA_TAP_SERVICES, Gaia
        try:
            _GAIA_MIRROR = TapPlus(url='https://gaia.ari.uni-heidelberg.de/tap')
        except Exception:
            pass
        try:
            from astroquery.gaia import GaiaClass
            Gaia = GaiaClass()
            try:
                Gaia.ROW_LIMIT = -1
            except Exception:
                pass
        except Exception:
            pass
        _GAIA_TAP_SERVICES = [('ARI mirror', _GAIA_MIRROR), ('ESA', Gaia)]
    
    def warn(message):
        text = f'WARNING: {message}'
        _WARNINGS.append(text)
        print(text)
    
    def _run_with_hard_timeout(fn, timeout_s, label='network operation'):
        box = {}
        done = threading.Event()
    
        def _runner():
            try:
                box['value'] = fn()
            except BaseException as exc:
                box['error'] = exc
            finally:
                done.set()
        threading.Thread(target=_runner, daemon=True).start()
        if not done.wait(timeout_s):
            raise TransientNetworkError(f'{label} exceeded {timeout_s}s; treating as a stalled connection')
        if 'error' in box:
            raise box['error']
        return box.get('value')
    
    def gaia_tap_results(query, async_query=False, attempts_per_service=1, label='Gaia TAP query', prefer=None):
        global _GAIA_TAP_SERVICES
        last_error = None
        services = list(_GAIA_TAP_SERVICES)
        if prefer is not None:
            services = sorted(services, key=lambda item: 0 if item[0] == prefer else 1)
        for service_name, service in services:
            for attempt in range(1, attempts_per_service + 1):
                use_async = bool(async_query and service_name == 'ESA')
                mode = 'async' if use_async else 'sync'
                print(f'{label}: starting {service_name} Gaia {mode} attempt {attempt}/{attempts_per_service}...', flush=True)
                try:
    
                    def _execute_tap():
                        job = service.launch_job_async(query, dump_to_file=False) if use_async else service.launch_job(query, dump_to_file=False)
                        return job.get_results()
                    result = _run_with_hard_timeout(_execute_tap, GAIA_TAP_TIMEOUT, f'{service_name} Gaia {mode} query')
                    print(f'{label}: {service_name} Gaia succeeded with {len(result)} rows.', flush=True)
                    _GAIA_TAP_SERVICES = [(service_name, service)] + [item for item in _GAIA_TAP_SERVICES if item[0] != service_name]
                    return result
                except Exception as exc:
                    last_error = exc
                    print(f'{label}: {service_name} Gaia attempt {attempt}/{attempts_per_service} failed: {exc}', flush=True)
                    if attempt < attempts_per_service:
                        time.sleep(3 * attempt)
        raise TransientNetworkError(f'All Gaia TAP services failed: {last_error}') from last_error
    
    def fmt_elapsed(seconds):
        seconds = int(round(seconds))
        return f'{seconds // 60:02d}:{seconds % 60:02d}'
    
    def atomic_to_csv(df, path):
        tmp = str(path) + '.tmp'
        df.to_csv(tmp, index=False)
        os.replace(tmp, path)
    
    def clean_identifier(value):
        if value is None:
            return np.nan
        try:
            if pd.isna(value):
                return np.nan
        except Exception:
            pass
        s = str(value).strip()
        if s == '' or s.lower() in {'nan', 'none', 'null', '--', 'masked', '<na>'}:
            return np.nan
        return s
    
    def parse_first_integer(value):
        s = clean_identifier(value)
        if pd.isna(s):
            return np.nan
        match = re.search('\\d+', str(s))
        return match.group(0) if match else np.nan
    
    def parse_first_float_from_arraylike(value):
        if value is None:
            return np.nan
        if isinstance(value, (list, tuple, np.ndarray)):
            try:
                arr = np.ma.filled(np.ma.asarray(value, dtype=float), np.nan).ravel()
                return float(arr[0]) if arr.size else np.nan
            except Exception:
                return np.nan
        if isinstance(value, str):
            match = re.search('[-+]?\\d*\\.?\\d+(?:[eE][-+]?\\d+)?', value)
            return float(match.group(0)) if match else np.nan
        try:
            result = float(value)
            return result
        except Exception:
            return np.nan
    
    def coerce_int64_ids(series):
        if pd.api.types.is_integer_dtype(series):
            return series.astype('Int64')
        if pd.api.types.is_float_dtype(series):
            if series.notna().any():
                raise ValueError('Gaia identifiers arrived as floating-point values; read them as strings or integers to preserve their digits.')
            return pd.Series(pd.NA, index=series.index, dtype='Int64')
    
        def parse(value):
            text = clean_identifier(value)
            if pd.isna(text):
                return pd.NA
            if re.fullmatch('[+]?[0-9]+', str(text)):
                return int(text)
            raise ValueError(f'Invalid integer source identifier: {text!r}')
        return pd.Series([parse(value) for value in series], index=series.index, dtype='Int64')
    
    def read_csv_ids_safe(path, **kwargs):
        try:
            header = pd.read_csv(path, nrows=0)
            dtype = {c: 'string' for c in ID_STRING_COLUMNS if c in header.columns}
        except Exception:
            dtype = None
        return pd.read_csv(path, dtype=dtype, on_bad_lines='skip', low_memory=False, **kwargs)
    
    def as_numeric(series_or_value):
        result = pd.to_numeric(series_or_value, errors='coerce')
        if isinstance(result, pd.Series):
            return result.replace([np.inf, -np.inf], np.nan)
        return result if pd.notna(result) and np.isfinite(result) else np.nan
    
    def norm_col(name):
        return re.sub('[^a-z0-9]+', '', str(name).lower())
    
    def find_column(columns, any_terms=None, all_terms=None, exclude_terms=None, preferred=None):
        columns = list(columns)
        norms = {col: norm_col(col) for col in columns}
        any_terms = [norm_col(t) for t in any_terms or []]
        all_terms = [norm_col(t) for t in all_terms or []]
        exclude_terms = [norm_col(t) for t in exclude_terms or []]
        preferred = [norm_col(t) for t in preferred or []]
        candidates = []
        for col, ncol in norms.items():
            if exclude_terms and any((term in ncol for term in exclude_terms)):
                continue
            if all_terms and (not all((term in ncol for term in all_terms))):
                continue
            if any_terms and (not any((term in ncol for term in any_terms))):
                continue
            candidates.append(col)
        if not candidates:
            return None
    
        def score(col):
            ncol = norms[col]
            score_value = 0
            for rank, pref in enumerate(preferred):
                if pref == ncol:
                    score_value += 1000 - rank
                elif pref in ncol:
                    score_value += 500 - rank
            if 'error' in ncol or 'err' in ncol or 'uncert' in ncol:
                score_value -= 100
            return score_value
        return sorted(candidates, key=score, reverse=True)[0]
    
    def rename_case_insensitive(df, expected_columns):
        rename = {}
        lower_to_col = {str(c).lower(): c for c in df.columns}
        for expected in expected_columns:
            found = lower_to_col.get(expected.lower())
            if found is not None and found != expected:
                rename[found] = expected
        if rename:
            df = df.rename(columns=rename)
        return df
    
    def astropy_to_pandas(table):
        try:
            return table.to_pandas()
        except Exception:
            pass
        n = len(table)
        data = {}
        for name in table.colnames:
            col = table[name]
            shape = getattr(col, 'shape', (n,))
            if len(shape) > 1:
                cells = []
                for i in range(n):
                    cell = col[i]
                    if cell is np.ma.masked:
                        cells.append(np.nan)
                    elif hasattr(cell, 'filled'):
                        cells.append(np.asarray(np.ma.asarray(cell, dtype=float).filled(np.nan)))
                    else:
                        cells.append(np.asarray(cell))
                data[name] = pd.Series(cells, index=pd.RangeIndex(n), dtype=object)
                continue
            mask = getattr(col, 'mask', None)
            if mask is not None and np.ndim(mask) and np.any(mask):
                if np.issubdtype(col.dtype, np.integer):
                    data[name] = np.asarray(col.astype(float).filled(np.nan))
                elif np.issubdtype(col.dtype, np.floating):
                    data[name] = np.asarray(col.filled(np.nan))
                else:
                    data[name] = np.asarray(col.astype(object).filled(np.nan))
            else:
                data[name] = np.asarray(col)
        return pd.DataFrame(data)
    
    def decode_bytes_column(series):
        return series.map(lambda v: v.decode() if isinstance(v, (bytes, bytearray)) else v)
    
    def ensure_final_columns(df):
        df = df.copy()
        for col in FINAL_COLUMNS:
            if col not in df.columns:
                df[col] = np.nan
        return df[FINAL_COLUMNS]
    
    def wrap_angle(angle):
        if not np.isfinite(angle):
            return np.nan
        return (angle + np.pi) % (2.0 * np.pi) - np.pi
    
    def feature_dict_nan(n_points=0):
        return {'P': np.nan, 'logP': np.nan, 'amplitude': np.nan, 'A1': np.nan, 'R21': np.nan, 'phi21': np.nan, 'R31': np.nan, 'phi31': np.nan, 'n_points': n_points}
    
    class TransientNetworkError(Exception):
        """A failure that looks like connectivity, not like missing data. Never cached."""
    
    class NetworkDownError(Exception):
        """Raised after many consecutive transient failures: stop the class, keep progress."""
    _TRANSIENT_KEYWORDS = ('timed out', 'timeout', 'connection', 'temporar', 'unreachable', 'name resolution', 'getaddrinfo', 'reset by peer', 'broken pipe', 'remote end closed', 'max retries', 'bad gateway', 'service unavailable', 'gateway timeout', 'too many requests', 'ssl', 'eof occurred', 'network', 'refused', ' 429', ' 500', ' 502', ' 503', ' 504')
    
    def _is_transient_exception(exc):
        if isinstance(exc, (TransientNetworkError, socket.timeout, ConnectionError, TimeoutError)):
            return True
        if isinstance(exc, requests.exceptions.RequestException):
            response = getattr(exc, 'response', None)
            if response is not None and response.status_code in (404, 410):
                return False
            return True
        text = str(exc).lower()
        return any((keyword in text for keyword in _TRANSIENT_KEYWORDS))
    from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
    _cache_dir_ready = False
    
    def _ensure_cache_dir():
        global _cache_dir_ready
        if not _cache_dir_ready:
            try:
                Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)
            except Exception:
                pass
            _cache_dir_ready = True
    
    def _cache_path(key):
        safe = re.sub('[^A-Za-z0-9_.-]+', '_', str(key))
        return Path(CACHE_DIR) / f'{safe}.json'
    
    def cache_get_features(key):
        _ensure_cache_dir()
        path = _cache_path(key)
        if not path.exists():
            return None
        try:
            with open(path, 'r', encoding='utf-8') as handle:
                data = json.load(handle)
        except Exception:
            return None
        status = data.pop('_status', None)
        feats = {k: np.nan if v is None else v for k, v in data.items()}
        if status is None:
            informative = [v for k, v in feats.items() if k != 'n_points' and isinstance(v, (int, float))]
            if not informative or all((not np.isfinite(v) for v in informative)):
                return None
            status = 'ok'
        return (feats, status)
    
    def cache_set_features(key, feats, status='ok'):
        if feats is None:
            return
        _ensure_cache_dir()
        path = _cache_path(key)
        try:
            serializable = {'_status': status}
            for k, v in feats.items():
                if isinstance(v, float) and (not np.isfinite(v)):
                    serializable[k] = None
                elif isinstance(v, (np.floating, np.integer)):
                    serializable[k] = float(v)
                else:
                    serializable[k] = v
            tmp = str(path) + '.tmp'
            with open(tmp, 'w', encoding='utf-8') as handle:
                json.dump(serializable, handle)
            os.replace(tmp, path)
        except Exception:
            pass
    
    def parallel_map(worker, items, desc, max_workers=None, should_stop=None):
        from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
        items = iter(items)
        workers = max(1, int(max_workers or N_WORKERS))
        with ThreadPoolExecutor(max_workers=workers) as executor:
            pending = {}
    
            def refill():
                while len(pending) < workers and (should_stop is None or not should_stop()):
                    try:
                        item = next(items)
                    except StopIteration:
                        break
                    pending[executor.submit(worker, item)] = item
            refill()
            with make_progress(desc=desc, unit='star') as progress:
                while pending:
                    done, _ = wait(set(pending), return_when=FIRST_COMPLETED)
                    for future in done:
                        item = pending.pop(future)
                        try:
                            result = future.result()
                        except Exception as exc:
                            warn(f'{desc}: worker failed: {exc}')
                            result = None
                        progress.update(1)
                        yield (item, result)
                    refill()
    
    def _ensure_checkpoint_dir():
        Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
    
    def checkpoint_path(class_name):
        return Path(CHECKPOINT_DIR) / f'checkpoint_{class_name}.csv'
    
    def candidate_key_for_row(row):
        if str(row.get('star_type')) in ('EB', 'MIRA', 'SRV') and (not pd.isna(clean_identifier(row.get('ogle_id')))):
            return str(row.get('ogle_id'))
        sid = row.get('source_id')
        if not pd.isna(sid):
            return str(sid)
        return str(clean_identifier(row.get('ogle_id')))
    _checkpoint_lock = threading.Lock()
    
    def checkpoint_append_rows(class_name, rows):
        if not rows:
            return
        _ensure_checkpoint_dir()
        df = ensure_final_columns(pd.DataFrame([dict(r) for r in rows]))
        df['candidate_key'] = [candidate_key_for_row(r) for r in rows]
        path = checkpoint_path(class_name)
        with _checkpoint_lock:
            df.to_csv(path, mode='a', header=not path.exists(), index=False)
    
    def checkpoint_overwrite(class_name, df):
        _ensure_checkpoint_dir()
        df = df.copy()
        body = ensure_final_columns(df)
        body['candidate_key'] = [candidate_key_for_row(r) for _, r in body.iterrows()]
        atomic_to_csv(body, checkpoint_path(class_name))
    
    def load_checkpoint(class_name):
        path = checkpoint_path(class_name)
        empty = ensure_final_columns(pd.DataFrame())
        empty['candidate_key'] = pd.Series(dtype='string')
        if not path.exists():
            return empty
        try:
            raw = read_csv_ids_safe(path)
        except Exception as exc:
            warn(f'Could not read checkpoint {path} ({exc}); starting that class fresh.')
            return empty
        if raw.empty:
            return empty
        keys = raw['candidate_key'].astype(str) if 'candidate_key' in raw.columns else pd.Series('', index=raw.index)
        body = prepare_common_dataframe(raw)
        body['quality_flag'] = compute_quality_flags(body)
        body['candidate_key'] = keys.values
        body = body[~body['candidate_key'].isin(['', 'nan', '<NA>', 'None'])]
        body = body.drop_duplicates(subset=['candidate_key'], keep='first').reset_index(drop=True)
        return body
    
    def strict_clean_mask(df):
        if df.empty:
            return pd.Series(False, index=df.index)
        conf = as_numeric(df.get('confidence'))
        mask = conf.ge(MIN_CONFIDENCE).fillna(False)
        mask &= df.get('quality_flag', pd.Series('', index=df.index)).eq('good')
        return mask
    
    def strict_clean_subset(df):
        if df.empty:
            return df
        return df[strict_clean_mask(df)]
    
    def clean_count_from_checkpoint(class_name):
        ck = load_checkpoint(class_name)
        clean = strict_clean_subset(ck)
        if 'source_id' in clean.columns:
            clean = clean.drop_duplicates(subset=['source_id'])
        return (len(clean), set(ck['candidate_key'].astype(str)))
    
    def estimate_period_lomb_scargle(time, mag, period_range=None):
        time = np.asarray(time, dtype=float)
        mag = np.asarray(mag, dtype=float)
        mask = np.isfinite(time) & np.isfinite(mag)
        time = time[mask]
        mag = mag[mask]
        if len(time) < MIN_LIGHTCURVE_POINTS:
            return np.nan
        order = np.argsort(time)
        time = time[order]
        mag = mag[order]
        time = time - np.nanmin(time)
        baseline = np.nanmax(time) - np.nanmin(time)
        if not np.isfinite(baseline) or baseline <= 0:
            return np.nan
        if len(time) > MAX_POINTS_FOR_LOMB_SCARGLE:
            idx = np.sort(_rng.choice(len(time), MAX_POINTS_FOR_LOMB_SCARGLE, replace=False))
            time = time[idx]
            mag = mag[idx]
        unique_time = np.unique(time)
        if len(unique_time) > 3:
            cadence = np.nanmedian(np.diff(unique_time))
            min_period = max(0.02, 2.0 * cadence) if np.isfinite(cadence) and cadence > 0 else 0.02
        else:
            min_period = 0.02
        max_period = min(100.0, max(2.0 * min_period, 0.8 * baseline))
        if period_range is not None:
            lo, hi = period_range
            min_period = max(min_period, float(lo))
            max_period = min(max_period, float(hi))
        if not np.isfinite(max_period) or max_period <= min_period:
            return np.nan
        y = mag - np.nanmedian(mag)
        if np.nanstd(y) <= 0:
            return np.nan
        try:
            frequency, power = LombScargle(time, y).autopower(minimum_frequency=1.0 / max_period, maximum_frequency=1.0 / min_period, samples_per_peak=5, normalization='standard')
            if len(frequency) == 0 or np.all(~np.isfinite(power)):
                return np.nan
            best_frequency = frequency[np.nanargmax(power)]
            if not np.isfinite(best_frequency) or best_frequency <= 0:
                return np.nan
            return float(1.0 / best_frequency)
        except Exception:
            return np.nan
    
    def extract_lightcurve_features(time, mag, period=None, mag_err=None, n_harmonics=3, period_range=None):
        try:
            time = np.asarray(time, dtype=float)
            mag = np.asarray(mag, dtype=float)
            base_mask = np.isfinite(time) & np.isfinite(mag)
            if mag_err is not None:
                mag_err = np.asarray(mag_err, dtype=float)
                if mag_err.shape == time.shape and np.any(np.isfinite(mag_err) & (mag_err > 0)):
                    mask = base_mask
                    mag_err = mag_err[mask]
                else:
                    mask = base_mask
                    mag_err = None
            else:
                mask = base_mask
            time = time[mask]
            mag = mag[mask]
            n_points = int(len(time))
            if n_points < max(MIN_LIGHTCURVE_POINTS, 2 * n_harmonics + 3):
                return feature_dict_nan(n_points=n_points)
            if period is not None and np.isfinite(period) and (float(period) > 0):
                P = float(period)
            else:
                P = estimate_period_lomb_scargle(time, mag, period_range=period_range)
            if not np.isfinite(P) or P <= 0:
                return feature_dict_nan(n_points=n_points)
            amplitude = float(np.nanpercentile(mag, 95) - np.nanpercentile(mag, 5))
            phase = time / P % 1.0
            finite = np.isfinite(phase) & np.isfinite(mag)
            phase = phase[finite]
            y = mag[finite]
            if mag_err is not None:
                sigma = mag_err[finite]
            else:
                sigma = None
            if len(y) < max(MIN_LIGHTCURVE_POINTS, 2 * n_harmonics + 3):
                return feature_dict_nan(n_points=n_points)
            design = [np.ones_like(phase)]
            for k in range(1, n_harmonics + 1):
                arg = 2.0 * np.pi * k * phase
                design.append(np.sin(arg))
                design.append(np.cos(arg))
            X = np.column_stack(design)
            good_sigma = np.isfinite(sigma) & (sigma > 0) if sigma is not None else None
            if sigma is not None and np.any(good_sigma):
                sigma_median = np.nanmedian(sigma[good_sigma])
                sigma_clean = np.where(good_sigma, sigma, sigma_median)
                w = 1.0 / np.clip(sigma_clean, sigma_median * 0.05, np.inf)
                X_fit = X * w[:, None]
                y_fit = y * w
            else:
                X_fit = X
                y_fit = y
            coeff, _, _, _ = np.linalg.lstsq(X_fit, y_fit, rcond=None)
            amps = []
            phis = []
            for k in range(n_harmonics):
                a_k = coeff[1 + 2 * k]
                b_k = coeff[2 + 2 * k]
                amps.append(float(np.sqrt(a_k ** 2 + b_k ** 2)))
                phis.append(float(np.arctan2(b_k, a_k)))
            A1 = amps[0]
            if not np.isfinite(A1) or A1 <= 0:
                out = feature_dict_nan(n_points=n_points)
                out.update({'P': P, 'logP': np.log10(P), 'amplitude': amplitude, 'A1': A1})
                return out
            return {'P': P, 'logP': float(np.log10(P)), 'amplitude': amplitude, 'A1': A1, 'R21': float(amps[1] / A1), 'phi21': float(wrap_angle(phis[1] - 2.0 * phis[0])), 'R31': float(amps[2] / A1), 'phi31': float(wrap_angle(phis[2] - 3.0 * phis[0])), 'n_points': n_points}
        except Exception:
            return feature_dict_nan(n_points=0)
    
    def add_gaia_derived_features(df):
        df = df.copy()
        for col in ['phot_g_mean_mag', 'bp_rp', 'parallax', 'parallax_error', 'ag_gspphot', 'ebpminrp_gspphot', 'skewness', 'stetson_J', 'P', 'amplitude', 'A1', 'R21', 'phi21', 'R31', 'phi31', 'n_points', 'confidence']:
            if col in df.columns:
                df[col] = as_numeric(df[col])
        ag_calc = df.get('ag_gspphot', pd.Series(np.nan, index=df.index)).fillna(0)
        ebp_calc = df.get('ebpminrp_gspphot', pd.Series(np.nan, index=df.index)).fillna(0)
        parallax = df.get('parallax', pd.Series(np.nan, index=df.index))
        phot_g = df.get('phot_g_mean_mag', pd.Series(np.nan, index=df.index))
        bp_rp = df.get('bp_rp', pd.Series(np.nan, index=df.index))
        valid_mg = phot_g.notna() & parallax.notna() & (parallax > 0)
        df['MG'] = np.nan
        df.loc[valid_mg, 'MG'] = phot_g.loc[valid_mg] + 5.0 * np.log10(parallax.loc[valid_mg] / 1000.0) + 5.0 - ag_calc.loc[valid_mg]
        df['BP_RP0'] = bp_rp - ebp_calc
        P = df.get('P', pd.Series(np.nan, index=df.index))
        valid_period = P.notna() & (P > 0)
        df['logP'] = np.nan
        df.loc[valid_period, 'logP'] = np.log10(P.loc[valid_period])
        df['cepheid_wesenheit_g'] = df['MG'] - CEPHEID_WESENHEIT_COLOR_COEFF * df['BP_RP0']
        df['delta_wesenheit_cepheid'] = np.nan
        valid_delta = df['cepheid_wesenheit_g'].notna() & df['logP'].notna() & (CEPHEID_PL_SIGMA > 0)
        df.loc[valid_delta, 'delta_wesenheit_cepheid'] = (df.loc[valid_delta, 'cepheid_wesenheit_g'] - (CEPHEID_PL_SLOPE * df.loc[valid_delta, 'logP'] + CEPHEID_PL_INTERCEPT)) / CEPHEID_PL_SIGMA
        return df
    
    def prepare_common_dataframe(df):
        df = df.copy()
        df = rename_case_insensitive(df, FINAL_COLUMNS + ML_FEATURES)
        if 'source_id' in df.columns:
            df['source_id'] = coerce_int64_ids(df['source_id'])
        if 'tic_id' in df.columns:
            df['tic_id'] = df['tic_id'].map(clean_identifier)
        if 'ogle_id' in df.columns:
            df['ogle_id'] = df['ogle_id'].map(clean_identifier)
        for col in ['ra', 'dec']:
            if col in df.columns:
                df[col] = as_numeric(df[col])
        df = add_gaia_derived_features(df)
        return ensure_final_columns(df)
    
    def compute_quality_flags(df):
        if df.empty:
            return pd.Series(dtype=object)
        flags = pd.Series([[] for _ in range(len(df))], index=df.index)
    
        def add(mask, name):
            mask = mask.fillna(True) if hasattr(mask, 'fillna') else mask
            for idx in df.index[mask]:
                flags.at[idx].append(name)
        num = {col: as_numeric(df.get(col, pd.Series(np.nan, index=df.index))) for col in ['ra', 'dec', 'phot_g_mean_mag', 'bp_rp', 'parallax', 'parallax_error', 'ag_gspphot', 'ebpminrp_gspphot', 'skewness', 'stetson_J', 'MG', 'BP_RP0', 'delta_wesenheit_cepheid', 'P', 'logP', 'amplitude', 'A1', 'R21', 'phi21', 'R31', 'phi31', 'n_points']}
        source_id = df.get('source_id', pd.Series(np.nan, index=df.index))
        add(source_id.isna(), 'missing_gaia')
        add(num['ra'].isna() | num['dec'].isna(), 'missing_coords')
        add(num['phot_g_mean_mag'].isna() | num['bp_rp'].isna(), 'missing_photometry')
        bad_parallax = num['parallax'].isna() | (num['parallax'] <= 0) | (num['parallax_error'].isna() | (num['parallax_error'] <= 0))
        add(bad_parallax, 'bad_parallax')
        with np.errstate(divide='ignore', invalid='ignore'):
            ratio = num['parallax_error'] / num['parallax']
        add(~bad_parallax & (ratio > MAX_PARALLAX_ERROR_RATIO), 'large_parallax_error')
        add(num['ag_gspphot'].isna() | num['ebpminrp_gspphot'].isna(), 'missing_extinction')
        add(num['skewness'].isna() | num['stetson_J'].isna(), 'missing_vari_stats')
        add(num['MG'].isna(), 'missing_MG')
        add(num['BP_RP0'].isna(), 'missing_color')
        add(num['P'].isna() | (num['P'] <= 0) | num['logP'].isna(), 'missing_period')
        add(num['amplitude'].isna() | num['A1'].isna(), 'missing_amplitude')
        add(num['delta_wesenheit_cepheid'].isna(), 'missing_delta_wesenheit')
        add(num['R21'].isna() | num['R31'].isna(), 'missing_fourier_shape')
        add(num['n_points'].isna() | (num['n_points'] <= 0), 'missing_npoints')
        return flags.map(lambda f: ';'.join(f) if f else 'good')
    
    def gaia_side_gate_mask(df):
        if df.empty:
            return pd.Series(False, index=df.index)
        num = {col: as_numeric(df.get(col, pd.Series(np.nan, index=df.index))) for col in ['ra', 'dec', 'phot_g_mean_mag', 'bp_rp', 'parallax', 'parallax_error', 'ag_gspphot', 'ebpminrp_gspphot', 'skewness', 'stetson_J']}
        mask = df.get('source_id', pd.Series(np.nan, index=df.index)).notna()
        for col in ['ra', 'dec', 'phot_g_mean_mag', 'bp_rp', 'ag_gspphot', 'ebpminrp_gspphot', 'skewness', 'stetson_J']:
            mask &= num[col].notna()
        mask &= num['parallax'].notna() & (num['parallax'] > 0) & num['parallax_error'].notna() & (num['parallax_error'] > 0)
        with np.errstate(divide='ignore', invalid='ignore'):
            ratio = num['parallax_error'] / num['parallax']
        mask &= ratio.le(MAX_PARALLAX_ERROR_RATIO).fillna(False)
        return mask.fillna(False)
    
    def finalize_candidate_row(candidate):
        frame = prepare_common_dataframe(pd.DataFrame([dict(candidate)]))
        frame['quality_flag'] = compute_quality_flags(frame)
        frame = ensure_final_columns(frame)
        return frame.iloc[0]
    
    def row_is_clean(row):
        try:
            confidence = float(row.get('confidence'))
        except Exception:
            return False
        if not np.isfinite(confidence) or confidence < MIN_CONFIDENCE:
            return False
        return row.get('quality_flag') == 'good'
    
    def query_gaia_pulsator_ids(gaia_class, top_n):
        star_type = GAIA_CLASS_MAP[gaia_class]
        if gaia_class == 'CEP':
            query = f'\n        SELECT TOP {int(top_n)}\n            cep.source_id AS source_id,\n            gs.random_index AS random_index\n        FROM gaiadr3.vari_cepheid AS cep\n        JOIN gaiadr3.gaia_source AS gs\n            ON cep.source_id = gs.source_id\n        WHERE gs.parallax > 0\n            AND cep.pf IS NOT NULL\n        ORDER BY random_index ASC\n        '
        else:
            query = f"\n        SELECT TOP {int(top_n)}\n            vc.source_id AS source_id,\n            vc.best_class_name AS best_class_name,\n            vc.best_class_score AS confidence,\n            gs.random_index AS random_index\n        FROM gaiadr3.vari_classifier_result AS vc\n        JOIN gaiadr3.gaia_source AS gs\n            ON vc.source_id = gs.source_id\n        WHERE vc.best_class_name = '{gaia_class}'\n            AND vc.best_class_score >= {MIN_CONFIDENCE}\n            AND gs.parallax > 0\n        ORDER BY best_class_score DESC, random_index ASC\n        "
        label = f'{star_type} ranked-ID query TOP {int(top_n)}'
        print(f'{star_type}: fetching ranked Gaia source IDs first (TOP {int(top_n)}).', flush=True)
        result = gaia_tap_results(query, async_query=False, attempts_per_service=1, label=label)
        ids = astropy_to_pandas(result)
        ids = rename_case_insensitive(ids, ['source_id', 'best_class_name', 'confidence', 'random_index'])
        if ids.empty:
            return ids
        if 'confidence' not in ids.columns:
            ids['confidence'] = 0.95
        ids['source_id'] = coerce_int64_ids(ids['source_id'])
        ids['confidence'] = pd.to_numeric(ids['confidence'], errors='coerce')
        ids['random_index'] = pd.to_numeric(ids.get('random_index'), errors='coerce')
        ids = ids.dropna(subset=['source_id', 'confidence'])
        ids = ids.drop_duplicates(subset=['source_id'], keep='first')
        ids['rank_order'] = np.arange(len(ids), dtype=int)
        print(f'{star_type}: ranked-ID query returned {len(ids)} candidates.', flush=True)
        return ids
    
    def fetch_gaia_pulsator_details(gaia_class, ranked_ids):
        star_type = GAIA_CLASS_MAP[gaia_class]
        spec_table = GAIA_SPECIALIZED_TABLES[gaia_class]
        if ranked_ids is None or ranked_ids.empty:
            return pd.DataFrame()
        select_specialized = ''.join((f',\n            sp.{column} AS {output_name}' for output_name, column in GAIA_SPEC_COLUMN_MAP.items()))
        ids = [int(x) for x in coerce_int64_ids(ranked_ids['source_id']).dropna().tolist()]
        chunks = [ids[i:i + GAIA_PULSATOR_DETAIL_BLOCK] for i in range(0, len(ids), GAIA_PULSATOR_DETAIL_BLOCK)]
        frames = []
        print(f'{star_type}: fetching Gaia/SOS details in {len(chunks)} chunks of up to {GAIA_PULSATOR_DETAIL_BLOCK}.', flush=True)
        for idx, chunk in enumerate(make_progress(chunks, desc=f'{star_type} Gaia detail chunks', unit='chunk'), start=1):
            id_list = ','.join((str(int(x)) for x in chunk))
            query = f'\n        SELECT\n            gs.source_id AS source_id,\n            gs.ra AS ra,\n            gs.dec AS dec,\n            gs.phot_g_mean_mag AS phot_g_mean_mag,\n            gs.bp_rp AS bp_rp,\n            gs.parallax AS parallax,\n            gs.parallax_error AS parallax_error,\n            ap.ag_gspphot AS ag_gspphot,\n            ap.ebpminrp_gspphot AS ebpminrp_gspphot,\n            vs.skewness_mag_g_fov AS skewness,\n            vs.stetson_mag_g_fov AS stetson_J{select_specialized}\n        FROM gaiadr3.gaia_source AS gs\n        LEFT JOIN gaiadr3.astrophysical_parameters AS ap\n            ON gs.source_id = ap.source_id\n        LEFT JOIN gaiadr3.vari_summary AS vs\n            ON gs.source_id = vs.source_id\n        LEFT JOIN {spec_table} AS sp\n            ON gs.source_id = sp.source_id\n        WHERE gs.source_id IN ({id_list})\n        '
            label = f'{star_type} detail chunk {idx}/{len(chunks)}'
            result = gaia_tap_results(query, async_query=False, attempts_per_service=1, label=label)
            part = astropy_to_pandas(result)
            part = rename_case_insensitive(part, ['source_id', 'ra', 'dec', 'phot_g_mean_mag', 'bp_rp', 'parallax', 'parallax_error', 'ag_gspphot', 'ebpminrp_gspphot', 'skewness', 'stetson_J'] + list(GAIA_SPEC_COLUMN_MAP.keys()))
            if not part.empty:
                frames.append(part)
        if not frames:
            return pd.DataFrame()
        detail = pd.concat(frames, ignore_index=True).drop_duplicates(subset=['source_id'], keep='first')
        detail['source_id'] = coerce_int64_ids(detail['source_id'])
        ranked = ranked_ids.copy()
        ranked['source_id'] = coerce_int64_ids(ranked['source_id'])
        merged = ranked.merge(detail, on='source_id', how='left')
        merged = merged.sort_values('rank_order', kind='mergesort').drop(columns=['rank_order'], errors='ignore')
        if 'A1' in merged.columns:
            merged['A1'] = merged['A1'].map(parse_first_float_from_arraylike)
        return merged
    
    def query_gaia_pulsator_class(gaia_class, top_n):
        star_type = GAIA_CLASS_MAP[gaia_class]
        raw_path = RAW_DIR / 'gaia_rrlyr_raw.csv' if star_type == 'RRLYR' else RAW_DIR / f'gaia_{star_type.lower()}_raw.csv'
        try:
            ranked_ids = query_gaia_pulsator_ids(gaia_class, top_n)
            if ranked_ids is None or ranked_ids.empty:
                warn(f'{star_type}: ranked Gaia query returned no candidates.')
                return pd.DataFrame()
            df = fetch_gaia_pulsator_details(gaia_class, ranked_ids)
            df = rename_case_insensitive(df, ['source_id', 'confidence', 'ra', 'dec', 'phot_g_mean_mag', 'bp_rp', 'parallax', 'parallax_error', 'ag_gspphot', 'ebpminrp_gspphot', 'skewness', 'stetson_J'] + list(GAIA_SPEC_COLUMN_MAP.keys()))
            atomic_to_csv(df, raw_path)
            print(f'Saved {raw_path}: {len(df)} rows', flush=True)
            return df
        except Exception as exc:
            warn(f'Failed Gaia query for {star_type}: {exc}')
            return None
    
    def fetch_gaia_source_features_by_source_ids(source_ids):
        ids = pd.Series(list(source_ids)).dropna()
        ids = coerce_int64_ids(ids).dropna().drop_duplicates().tolist()
        if not ids:
            return pd.DataFrame()
        chunks = [ids[i:i + GAIA_BATCH_SIZE] for i in range(0, len(ids), GAIA_BATCH_SIZE)]
        frames = []
        refresh_gaia_tap_services()
        for idx, chunk in enumerate(make_progress(chunks, desc='Gaia auxiliary chunks', unit='chunk'), start=1):
            id_list = ','.join((str(int(x)) for x in chunk))
            query = f'\n        SELECT\n            gs.source_id AS source_id,\n            gs.ra AS gaia_ra,\n            gs.dec AS gaia_dec,\n            gs.phot_g_mean_mag AS phot_g_mean_mag,\n            gs.bp_rp AS bp_rp,\n            gs.parallax AS parallax,\n            gs.parallax_error AS parallax_error,\n            ap.ag_gspphot AS ag_gspphot,\n            ap.ebpminrp_gspphot AS ebpminrp_gspphot,\n            vs.skewness_mag_g_fov AS skewness,\n            vs.stetson_mag_g_fov AS stetson_J\n        FROM gaiadr3.gaia_source AS gs\n        LEFT JOIN gaiadr3.astrophysical_parameters AS ap\n            ON gs.source_id = ap.source_id\n        LEFT JOIN gaiadr3.vari_summary AS vs\n            ON gs.source_id = vs.source_id\n        WHERE gs.source_id IN ({id_list})\n        '
            result = None
            for _attempt in range(1, 4):
                try:
                    result = gaia_tap_results(query, label=f'Gaia auxiliary chunk {idx}/{len(chunks)}')
                    break
                except Exception as exc:
                    if _attempt >= 3:
                        raise TransientNetworkError(f'Gaia source_id batch query failed after retries: {exc}')
                    refresh_gaia_tap_services()
                    time.sleep(2 * _attempt)
            part = astropy_to_pandas(result)
            part = rename_case_insensitive(part, ['source_id', 'gaia_ra', 'gaia_dec', 'phot_g_mean_mag', 'bp_rp', 'parallax', 'parallax_error', 'ag_gspphot', 'ebpminrp_gspphot', 'skewness', 'stetson_J'])
            frames.append(part)
        if not frames:
            return pd.DataFrame()
        return pd.concat(frames, ignore_index=True).drop_duplicates(subset=['source_id'])
    GAIA_AUX_CACHE_PATH = Path(CHECKPOINT_DIR) / 'cache_gaia_aux.csv'
    _gaia_aux_cache = None
    GAIA_AUX_COLUMNS = ['source_id', 'gaia_ra', 'gaia_dec', 'phot_g_mean_mag', 'bp_rp', 'parallax', 'parallax_error', 'ag_gspphot', 'ebpminrp_gspphot', 'skewness', 'stetson_J']
    
    def _load_gaia_aux_cache():
        global _gaia_aux_cache
        if _gaia_aux_cache is not None:
            return _gaia_aux_cache
        _gaia_aux_cache = {}
        if GAIA_AUX_CACHE_PATH.exists():
            try:
                cached = read_csv_ids_safe(GAIA_AUX_CACHE_PATH)
                for _, row in cached.iterrows():
                    sid = clean_identifier(row.get('source_id'))
                    if not pd.isna(sid):
                        _key = str(sid)
                        _key = _key[:-2] if _key.endswith('.0') else _key
                        _gaia_aux_cache[_key] = row.to_dict()
            except Exception:
                _gaia_aux_cache = {}
        return _gaia_aux_cache
    
    def fetch_gaia_aux_cached(source_ids):
        cache = _load_gaia_aux_cache()
        ids = pd.Series(list(source_ids)).dropna()
        ids = coerce_int64_ids(ids).dropna().drop_duplicates().tolist()
        missing = [i for i in ids if str(i) not in cache]
        if missing:
            fetched = fetch_gaia_source_features_by_source_ids(missing)
            if not fetched.empty:
                fetched['source_id'] = coerce_int64_ids(fetched['source_id'])
                new_rows = []
                for _, row in fetched.iterrows():
                    record = {col: row.get(col) for col in GAIA_AUX_COLUMNS}
                    record['source_id'] = str(row['source_id'])
                    cache[record['source_id']] = record
                    new_rows.append(record)
                if new_rows:
                    _ensure_checkpoint_dir()
                    cache_frame = pd.DataFrame(list(cache.values()))
                    for col in GAIA_AUX_COLUMNS:
                        if col not in cache_frame.columns:
                            cache_frame[col] = np.nan
                    cache_frame = cache_frame[GAIA_AUX_COLUMNS].drop_duplicates('source_id', keep='last')
                    with _checkpoint_lock:
                        atomic_to_csv(cache_frame, GAIA_AUX_CACHE_PATH)
        rows = [cache[str(i)] for i in ids if str(i) in cache]
        return pd.DataFrame(rows, columns=GAIA_AUX_COLUMNS)
    
    def fetch_block_aux_or_skip(source_ids, state):
        try:
            aux = fetch_gaia_aux_cached(source_ids)
            state['consec'] = 0
            return (aux, False)
        except (TransientNetworkError, NetworkDownError) as exc:
            state['consec'] = state.get('consec', 0) + 1
            warn(f"{state.get('label', 'class')}: Gaia auxiliary fetch failed for this block ({str(exc)[:140]}); skipping it for now (its stars stay pending and resume next run) and continuing. [{state['consec']}/{AUX_ABORT_AFTER} consecutive boundary failures]")
            refresh_gaia_tap_services()
            if state['consec'] >= AUX_ABORT_AFTER:
                raise NetworkDownError(f'{AUX_ABORT_AFTER} consecutive Gaia auxiliary-fetch failures — the network looks down')
            time.sleep(min(30, 3 * state['consec']))
            return (None, True)
        except (ValueError, OSError) as exc:
            warn(f"{state.get('label', 'class')}: recovered from a stdout/stream error at the batch boundary ({str(exc)[:120]}); skipping this block and continuing.")
            return (None, True)
    
    def gaia_crossmatch_radec(ra, dec, radius_arcsec=GAIA_CROSSMATCH_RADIUS_ARCSEC):
        try:
            ra = float(ra)
            dec = float(dec)
            if not np.isfinite(ra) or not np.isfinite(dec):
                return {}
            radius_deg = radius_arcsec / 3600.0
            query = f"\n        SELECT TOP 1\n            gs.source_id AS source_id,\n            DISTANCE(POINT('ICRS', gs.ra, gs.dec), POINT('ICRS', {ra}, {dec})) AS dist_deg\n        FROM gaiadr3.gaia_source AS gs\n        WHERE 1 = CONTAINS(\n            POINT('ICRS', gs.ra, gs.dec),\n            CIRCLE('ICRS', {ra}, {dec}, {radius_deg})\n        )\n        ORDER BY dist_deg ASC\n        "
            result = gaia_tap_results(query, label='Per-star Gaia cone crossmatch')
            if len(result) == 0:
                return {}
            df = astropy_to_pandas(result)
            return df.iloc[0].to_dict()
        except Exception as exc:
            if _is_transient_exception(exc):
                raise TransientNetworkError(str(exc))
            return {}
    
    def gaia_batch_crossmatch(block):
        upload = Table.from_pandas(block[['ogle_id', 'ra', 'dec']].reset_index(drop=True))
        result = XMatch.query(cat1=upload, colRA1='ra', colDec1='dec', cat2='vizier:I/355/gaiadr3', max_distance=GAIA_CROSSMATCH_RADIUS_ARCSEC * u.arcsec)
        df = astropy_to_pandas(result)
        if df.empty:
            return pd.DataFrame(columns=['ogle_id', 'source_id', 'match_dist_arcsec'])
        source_col = 'Source' if 'Source' in df.columns else find_column(df.columns, any_terms=['source'], preferred=['Source', 'source_id'])
        dist_col = 'angDist' if 'angDist' in df.columns else find_column(df.columns, any_terms=['angdist', 'dist'], preferred=['angDist'])
        if source_col is None or dist_col is None or 'ogle_id' not in df.columns:
            raise RuntimeError(f'Unexpected XMatch result columns: {df.columns.tolist()}')
        out = pd.DataFrame({'ogle_id': decode_bytes_column(df['ogle_id']).astype(str), 'source_id': coerce_int64_ids(df[source_col]), 'match_dist_arcsec': as_numeric(df[dist_col])})
        out = out.sort_values('match_dist_arcsec', kind='mergesort')
        out = out.drop_duplicates(subset=['ogle_id'], keep='first')
        return out.reset_index(drop=True)
    EB_CROSSMATCH_CACHE_PATH = Path(CHECKPOINT_DIR) / 'cache_eb_crossmatch.csv'
    _ogle_crossmatch_caches = {}
    
    def _load_ogle_crossmatch_cache(cache_path):
        key = str(cache_path)
        if key in _ogle_crossmatch_caches:
            return _ogle_crossmatch_caches[key]
        cache = {}
        if Path(cache_path).exists():
            try:
                cached = read_csv_ids_safe(cache_path)
                for _, row in cached.iterrows():
                    oid = clean_identifier(row.get('ogle_id'))
                    if pd.isna(oid):
                        continue
                    sid = clean_identifier(row.get('source_id'))
                    cache[str(oid)] = {'source_id': np.nan if pd.isna(sid) else str(sid), 'match_dist_arcsec': as_numeric(row.get('match_dist_arcsec'))}
            except Exception:
                cache = {}
        _ogle_crossmatch_caches[key] = cache
        return cache
    
    def _append_ogle_crossmatch_cache(cache_path, records):
        if not records:
            return
        _ensure_checkpoint_dir()
        df = pd.DataFrame(records, columns=['ogle_id', 'source_id', 'match_dist_arcsec'])
        with _checkpoint_lock:
            df.to_csv(cache_path, mode='a', header=not Path(cache_path).exists(), index=False)
    
    def ogle_crossmatch_cached(block, cache_path):
        cache = _load_ogle_crossmatch_cache(cache_path)
        need = block[~block['ogle_id'].astype(str).isin(cache.keys())]
        if not need.empty:
            matched = None
            try:
                matched = gaia_batch_crossmatch(need)
            except Exception as exc:
                warn(f'Batched CDS crossmatch unavailable ({str(exc)[:160]}); falling back to per-star crossmatching.')
            new_records = []
            if matched is not None:
                matched_map = {str(r['ogle_id']): r for _, r in matched.iterrows()}
                for oid in need['ogle_id'].astype(str):
                    row = matched_map.get(oid)
                    record = {'ogle_id': oid, 'source_id': str(row['source_id']) if row is not None and (not pd.isna(row['source_id'])) else np.nan, 'match_dist_arcsec': float(row['match_dist_arcsec']) if row is not None else np.nan}
                    cache[oid] = {'source_id': record['source_id'], 'match_dist_arcsec': record['match_dist_arcsec']}
                    new_records.append(record)
            else:
    
                def crossmatch_one(item):
                    oid, ra, dec = item
                    try:
                        hit = gaia_crossmatch_radec(ra, dec)
                        return ('ok', hit)
                    except TransientNetworkError:
                        return ('transient', None)
                items = list(zip(need['ogle_id'].astype(str), need['ra'], need['dec']))
                transient_streak = 0
                for item, outcome in parallel_map(crossmatch_one, items, desc='Per-star Gaia crossmatch'):
                    oid = item[0]
                    status, hit = outcome if outcome is not None else ('transient', None)
                    if status == 'transient':
                        transient_streak += 1
                        if transient_streak >= TRANSIENT_ABORT_AFTER:
                            _append_ogle_crossmatch_cache(cache_path, new_records)
                            raise TransientNetworkError('Per-star crossmatch: repeated network failures')
                        continue
                    transient_streak = 0
                    sid = hit.get('source_id') if hit else None
                    record = {'ogle_id': oid, 'source_id': str(int(sid)) if sid is not None and (not pd.isna(sid)) else np.nan, 'match_dist_arcsec': float(hit.get('dist_deg', np.nan)) * 3600.0 if hit else np.nan}
                    cache[oid] = {'source_id': record['source_id'], 'match_dist_arcsec': record['match_dist_arcsec']}
                    new_records.append(record)
            _append_ogle_crossmatch_cache(cache_path, new_records)
        return {oid: cache[oid] for oid in block['ogle_id'].astype(str) if oid in cache}
    
    def read_zhou_catalogue():
        path = Path(ZHOU_RAW_PATH)
        raw_text = None
        if path.exists() and (path.stat().st_size > 1000000):
            try:
                raw_text = path.read_text(encoding='utf-8')
                print(f'Zhou catalogue: reusing cached {path} ({path.stat().st_size // 1024} KB).')
            except Exception:
                raw_text = None
        if raw_text is None:
            try:
                print('Downloading Zhou TESS-validated DSCT/GDOR catalogue (one-time, cached afterwards)...')
                response = requests.get(ZHOU_URL, timeout=CATALOG_DOWNLOAD_TIMEOUT)
                response.raise_for_status()
                raw_text = response.text
                path.write_text(raw_text, encoding='utf-8')
                print(f'Saved {path}')
            except Exception as exc:
                warn(f'Failed to download Zhou catalogue: {exc}')
                return pd.DataFrame()
        lines = raw_text.splitlines()
        header_line = None
        for line in lines:
            stripped = line.lstrip()
            uncommented = stripped.lstrip('#').strip()
            if stripped.startswith('#') and re.match('^TIC\\s*,', uncommented, flags=re.IGNORECASE) and ('Gaia' in uncommented):
                header_line = uncommented
                break
        data_lines = [line for line in lines if line.strip() and (not line.lstrip().startswith('#'))]
        if not data_lines:
            warn('Zhou catalogue downloaded but no data rows were found.')
            return pd.DataFrame()
        if header_line:
            names = [name.strip() for name in header_line.split(',')]
        else:
            warn('Failed Zhou column inference: no commented header line found.')
            names = None
        parsed_rows = []
        malformed_rows = 0
        expected_n = len(names) if names else None
        for line in data_lines:
            try:
                row = next(csv.reader([line], skipinitialspace=True))
            except Exception:
                malformed_rows += 1
                continue
            if expected_n is not None:
                if len(row) < expected_n:
                    row = row + [np.nan] * (expected_n - len(row))
                    malformed_rows += 1
                elif len(row) > expected_n:
                    row = row[:expected_n - 1] + [','.join(row[expected_n - 1:]).strip()]
                    malformed_rows += 1
            parsed_rows.append(row)
        if not parsed_rows:
            warn('Zhou catalogue rows could not be parsed after download.')
            return pd.DataFrame()
        if names:
            df = pd.DataFrame(parsed_rows, columns=names)
        else:
            max_cols = max((len(row) for row in parsed_rows))
            parsed_rows = [row + [np.nan] * (max_cols - len(row)) for row in parsed_rows]
            df = pd.DataFrame(parsed_rows, columns=[f'col_{i}' for i in range(max_cols)])
        if malformed_rows:
            warn(f'Zhou catalogue had {malformed_rows} rows with non-standard field counts; repaired by padding short rows or merging extra fields into the final column.')
        return df
    
    def infer_zhou_columns(df):
        cols = list(df.columns)
        inferred = {'source_id': find_column(cols, any_terms=['gaia_dr3', 'source_id', 'sourceid', 'gaia'], preferred=['Gaia_DR3', 'source_id']), 'tic_id': find_column(cols, any_terms=['tic'], preferred=['TIC', 'tic_id']), 'type': find_column(cols, any_terms=['vartype', 'type', 'class', 'classification'], preferred=['VarType_Notes', 'class', 'type']), 'rank_score': find_column(cols, any_terms=['score', 'confidence', 'probability', 'prob'], exclude_terms=['error', 'err'], preferred=['Score_of_class', 'score_of_class', 'confidence']), 'tmag': find_column(cols, any_terms=['tmag'], exclude_terms=['error', 'err'], preferred=['Tmag']), 'ra': find_column(cols, any_terms=['ra'], preferred=['RA_deg', 'ra']), 'dec': find_column(cols, any_terms=['dec'], preferred=['Dec_deg', 'dec'])}
        print('Inferred Zhou columns:')
        print(inferred)
        essential = ['source_id', 'type']
        missing = [key for key in essential if inferred.get(key) is None]
        if missing:
            warn(f'Failed Zhou column inference for required columns {missing}. DSCT/GDOR rows cannot be built from Zhou.')
        return inferred
    
    def clean_zhou_label(value):
        s = str(value).strip().upper()
        exclude_terms = ['HYBRID', 'DSCT/GDOR', 'GDOR/DSCT', 'DSCT+GDOR', 'GDOR+DSCT', 'SXPHE', 'EB', 'ECL', 'UNCERTAIN', 'UNCLEAR', 'CANDIDATE', 'UNKNOWN', 'CONSTANT', 'TRANSIT', 'RECHECK']
        if any((term in s for term in exclude_terms)) or any((symbol in s for symbol in ['?', '+', '/', '|'])):
            return None
        has_dsct = 'DSCT' in s or 'DELTA' in s
        has_gdor = 'GDOR' in s or 'GAMMA' in s
        if has_dsct and (not has_gdor):
            return 'DSCT'
        if has_gdor and (not has_dsct):
            return 'GDOR'
        return None
    
    def build_zhou_candidate_table():
        zhou = read_zhou_catalogue()
        if zhou.empty:
            return pd.DataFrame()
        inferred = infer_zhou_columns(zhou)
        if inferred.get('source_id') is None or inferred.get('type') is None:
            return pd.DataFrame()
        rows = pd.DataFrame(index=zhou.index)
        rows['source_id'] = coerce_int64_ids(zhou[inferred['source_id']].map(parse_first_integer))
        rows['tic_id'] = zhou[inferred['tic_id']].map(parse_first_integer) if inferred.get('tic_id') else np.nan
        rows['ra'] = pd.to_numeric(zhou[inferred['ra']], errors='coerce') if inferred.get('ra') else np.nan
        rows['dec'] = pd.to_numeric(zhou[inferred['dec']], errors='coerce') if inferred.get('dec') else np.nan
        rows['star_type'] = zhou[inferred['type']].map(clean_zhou_label)
        rows['zhou_rank_score'] = pd.to_numeric(zhou[inferred['rank_score']], errors='coerce') if inferred.get('rank_score') else np.nan
        rows['zhou_tmag'] = pd.to_numeric(zhou[inferred['tmag']], errors='coerce') if inferred.get('tmag') else np.nan
        rows = rows[rows['star_type'].isin(['DSCT', 'GDOR'])].copy()
        rows = rows.dropna(subset=['source_id'])
        rows = rows.drop_duplicates(subset=['source_id'], keep='first')
        print(f"Zhou catalogue: {int((rows['star_type'] == 'DSCT').sum())} clean-labelled DSCT and {int((rows['star_type'] == 'GDOR').sum())} clean-labelled GDOR candidates.")
        return rows
    
    def extract_tess_arrays_from_lightkurve(lc):
        try:
            lc = lc.remove_nans()
        except Exception:
            pass
        time_values = np.asarray(getattr(lc.time, 'value', lc.time), dtype=float)
        flux_values = np.asarray(getattr(lc.flux, 'value', lc.flux), dtype=float)
        if hasattr(lc, 'flux_err') and lc.flux_err is not None:
            flux_err_values = np.asarray(getattr(lc.flux_err, 'value', lc.flux_err), dtype=float)
        else:
            flux_err_values = np.full_like(flux_values, np.nan, dtype=float)
        mask = np.isfinite(time_values) & np.isfinite(flux_values) & (flux_values > 0)
        time_values = time_values[mask]
        flux_values = flux_values[mask]
        flux_err_values = flux_err_values[mask]
        if len(time_values) < MIN_LIGHTCURVE_POINTS:
            return (None, None, None)
        median_flux = np.nanmedian(flux_values)
        if not np.isfinite(median_flux) or median_flux <= 0:
            return (None, None, None)
        mag = -2.5 * np.log10(flux_values / median_flux)
        mag_err = np.full_like(mag, np.nan, dtype=float)
        good_err = np.isfinite(flux_err_values) & (flux_err_values > 0)
        mag_err[good_err] = 1.0857362047581296 * flux_err_values[good_err] / flux_values[good_err]
        med = np.nanmedian(mag)
        mad = stats.median_abs_deviation(mag[np.isfinite(mag)], nan_policy='omit') if np.isfinite(mag).any() else np.nan
        if np.isfinite(mad) and mad > 0:
            clip_mask = np.abs(mag - med) < 8.0 * 1.4826 * mad
            time_values = time_values[clip_mask]
            mag = mag[clip_mask]
            mag_err = mag_err[clip_mask]
        return (time_values, mag, mag_err)
    
    def _pick_best_search_row(search):
        try:
            authors = [str(a) for a in search.table['author']]
        except Exception:
            return 0
        try:
            dist = np.asarray(search.table['distance'], dtype=float)
            if not np.any(np.isfinite(dist)):
                dist = None
        except Exception:
            dist = None
    
        def nearest(indices):
            if dist is None:
                return indices[0]
            return min(indices, key=lambda i: dist[i] if np.isfinite(dist[i]) else np.inf)
        for preferred in TESS_AUTHOR_PRIORITY:
            matches = [idx for idx, author in enumerate(authors) if author.upper() == preferred.upper()]
            if matches:
                return nearest(matches)
        return nearest(list(range(len(authors))))
    
    def _is_corrupt_cache_error(exc):
        text = str(exc).lower()
        return any((s in text for s in ('error in reading data product', 'empty or corrupt fits', 'is truncated', 'buffer is too small', 'header missing end card', 'no such card')))
    
    def _download_lightcurve(search_row):
        LIGHTCURVE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
        return search_row.download(quality_bitmask='default', download_dir=str(LIGHTCURVE_CACHE_DIR))
    
    def tess_lightcurve_features_fast(tic_id=None, ra=None, dec=None, period=None, period_range=None):
        tic_clean = parse_first_integer(tic_id)
        try:
            if not pd.isna(tic_clean):
                search = lk.search_lightcurve(f'TIC {tic_clean}', mission='TESS')
            elif ra is not None and dec is not None and np.isfinite(float(ra)) and np.isfinite(float(dec)):
                target = SkyCoord(float(ra) * u.deg, float(dec) * u.deg, frame='icrs')
                search = lk.search_lightcurve(target, mission='TESS', radius=2 * u.arcsec)
            else:
                return (feature_dict_nan(n_points=0), 'no_data')
        except Exception as exc:
            if _is_transient_exception(exc):
                raise TransientNetworkError(f'TESS search failed: {exc}')
            return (feature_dict_nan(n_points=0), 'no_data')
        if search is None or len(search) == 0:
            return (feature_dict_nan(n_points=0), 'no_data')
        idx = _pick_best_search_row(search)
        try:
            lc = _download_lightcurve(search[idx])
        except Exception as exc:
            if _is_corrupt_cache_error(exc):
                return (feature_dict_nan(n_points=0), 'no_data')
            if _is_transient_exception(exc):
                raise TransientNetworkError(f'TESS download failed: {exc}')
            return (feature_dict_nan(n_points=0), 'no_data')
        if lc is None:
            return (feature_dict_nan(n_points=0), 'no_data')
        time_values, mag, mag_err = extract_tess_arrays_from_lightkurve(lc)
        if time_values is None:
            return (feature_dict_nan(n_points=0), 'no_data')
        return (extract_lightcurve_features(time_values, mag, period=period, mag_err=mag_err, period_range=period_range), 'ok')
    
    def parse_ogle_ident_text(text):
        ids, subtypes, ra_strings, dec_strings = ([], [], [], [])
        for line in text.splitlines():
            if not line.strip() or line.lstrip().startswith('#'):
                continue
            parts = line.split()
            if len(parts) < 4:
                continue
            ids.append(parts[0])
            subtypes.append(parts[1])
            ra_strings.append(parts[2])
            dec_strings.append(parts[3])
        if not ids:
            return pd.DataFrame()
        try:
            coords = SkyCoord(ra_strings, dec_strings, unit=(u.hourangle, u.deg), frame='icrs')
            return pd.DataFrame({'ogle_id': ids, 'ogle_subtype': subtypes, 'ra': coords.ra.deg, 'dec': coords.dec.deg})
        except Exception:
            rows = []
            for oid, subtype, ra_s, dec_s in zip(ids, subtypes, ra_strings, dec_strings):
                try:
                    coord = SkyCoord(ra_s, dec_s, unit=(u.hourangle, u.deg), frame='icrs')
                    rows.append({'ogle_id': oid, 'ogle_subtype': subtype, 'ra': float(coord.ra.deg), 'dec': float(coord.dec.deg)})
                except Exception:
                    continue
            return pd.DataFrame(rows)
    
    def parse_ogle_ecl_text(text):
        rows = []
        for line in text.splitlines():
            if not line.strip() or line.lstrip().startswith('#'):
                continue
            parts = line.split()
            if len(parts) < 7:
                continue
            rows.append({'ogle_id': parts[0], 'I_max': pd.to_numeric(parts[1], errors='coerce'), 'V_max': pd.to_numeric(parts[2], errors='coerce'), 'period': pd.to_numeric(parts[3], errors='coerce'), 'epoch_primary': pd.to_numeric(parts[4], errors='coerce'), 'depth_primary': pd.to_numeric(parts[5], errors='coerce'), 'depth_secondary': pd.to_numeric(parts[6], errors='coerce')})
        return pd.DataFrame(rows)
    
    def load_optional_local_ogle_catalogue():
        local_path = Path(DATA_DIR / 'ogle_eb_catalog.csv')
        if not local_path.exists():
            return pd.DataFrame()
        try:
            local = pd.read_csv(local_path)
            required = {'ogle_id', 'ra', 'dec', 'period', 'lc_file'}
            if not required.issubset(local.columns):
                warn(f'Local {local_path} exists but is missing required columns {sorted(required - set(local.columns))}.')
                return pd.DataFrame()
            local['region'] = 'local'
            print(f'Loaded optional local OGLE catalogue {local_path}: {len(local)} rows')
            return local
        except Exception as exc:
            warn(f'Failed reading optional local OGLE catalogue: {exc}')
            return pd.DataFrame()
    
    def load_ogle_eb_catalogue():
        want_regions = {region for region, _, _ in OGLE_REGIONS}
        if Path(OGLE_CATALOG_CACHE).exists():
            try:
                cached = pd.read_csv(OGLE_CATALOG_CACHE)
                have_regions = set(cached.get('region', pd.Series(dtype=str)).astype(str))
                if want_regions.issubset(have_regions) and 'lc_file' in cached.columns:
                    print(f'OGLE EB catalogue: reusing cached {OGLE_CATALOG_CACHE} ({len(cached)} rows).')
                    local = load_optional_local_ogle_catalogue()
                    return pd.concat([local, cached], ignore_index=True) if not local.empty else cached
            except Exception:
                pass
        frames = []
        for region, base_url, lc_subdir in OGLE_REGIONS:
            try:
                print(f'Downloading OGLE {region.upper()} EB catalogue (one-time, cached afterwards)...')
                ecl_text = http_get_text_with_retry(f'{base_url}/ecl.dat', timeout=CATALOG_DOWNLOAD_TIMEOUT)
                ident_text = http_get_text_with_retry(f'{base_url}/ident.dat', timeout=CATALOG_DOWNLOAD_TIMEOUT)
                if ecl_text is None or ident_text is None:
                    warn(f'OGLE {region.upper()} EB catalogue files not found (404); skipping region.')
                    continue
                ecl_df = parse_ogle_ecl_text(ecl_text)
                ident_df = parse_ogle_ident_text(ident_text)
                merged = ecl_df.merge(ident_df, on='ogle_id', how='left')
                merged['region'] = region
                merged['lc_file'] = merged['ogle_id'].map(lambda oid: f'{base_url}/{lc_subdir}/{oid}.dat')
                frames.append(merged)
                print(f'Loaded OGLE {region.upper()} EB catalogue: {len(merged)} rows')
            except Exception as exc:
                warn(f'Failed to load OGLE {region.upper()} EB catalogue: {exc}')
        local = load_optional_local_ogle_catalogue()
        if not frames and local.empty:
            warn('No OGLE EB catalogue could be loaded (offline?). EB resumes on the next run.')
            return pd.DataFrame()
        if frames:
            public = pd.concat(frames, ignore_index=True)
            public = public.dropna(subset=['ogle_id', 'ra', 'dec', 'period', 'lc_file'])
            atomic_to_csv(public, OGLE_CATALOG_CACHE)
            print(f'Saved {OGLE_CATALOG_CACHE}: {len(public)} rows')
        else:
            public = pd.DataFrame()
        parts = [df for df in [local, public] if not df.empty]
        return pd.concat(parts, ignore_index=True)
    
    def rank_ogle_eb_candidates(raw):
        raw = raw.copy()
        for col in ['period', 'I_max', 'depth_primary', 'depth_secondary']:
            if col not in raw.columns:
                raw[col] = np.nan
            raw[col] = pd.to_numeric(raw[col], errors='coerce')
        raw = raw.dropna(subset=['ogle_id', 'ra', 'dec', 'lc_file'])
        raw = raw.drop_duplicates(subset=['ogle_id'], keep='first')
        raw['_has_period'] = (raw['period'].notna() & (raw['period'] > 0)).astype(int)
        raw['_region_priority'] = raw.get('region', pd.Series('zzz', index=raw.index)).map(OGLE_REGION_PRIORITY).fillna(9)
        raw['_depth_score'] = raw['depth_primary'].fillna(0) + raw['depth_secondary'].fillna(0)
        raw = raw.sort_values(['_has_period', '_region_priority', 'I_max', '_depth_score', 'ogle_id'], ascending=[False, True, True, False, True], kind='mergesort', na_position='last')
        raw = raw.drop(columns=['_has_period', '_region_priority', '_depth_score'], errors='ignore')
        return raw.reset_index(drop=True)
    
    def http_get_text_with_retry(url, timeout=REQUEST_TIMEOUT, attempts=OGLE_HTTP_ATTEMPTS, base_sleep=OGLE_HTTP_BASE_SLEEP):
        last = None
        for i in range(max(1, attempts)):
            try:
                response = requests.get(url, timeout=timeout)
                if response.status_code in (404, 410):
                    return None
                response.raise_for_status()
                return response.text
            except requests.exceptions.RequestException as exc:
                resp = getattr(exc, 'response', None)
                if resp is not None and getattr(resp, 'status_code', None) in (404, 410):
                    return None
                last = exc
                if i < attempts - 1:
                    time.sleep(base_sleep * 2 ** i + random.uniform(0, base_sleep))
        raise TransientNetworkError(f'GET {url} failed after {attempts} attempts: {last}')
    
    def read_ogle_lightcurve(lc_file):
        lc_file = str(lc_file)
        try:
            if re.match('^https?://', lc_file):
                text = http_get_text_with_retry(lc_file, timeout=REQUEST_TIMEOUT)
                if text is None:
                    return pd.DataFrame()
            else:
                text = Path(lc_file).read_text(encoding='utf-8')
        except TransientNetworkError:
            raise
        except OSError:
            return pd.DataFrame()
        try:
            df = pd.read_csv(io.StringIO(text), sep='\\s+|,', engine='python', names=['time', 'mag', 'mag_err'], comment='#')
            for col in ['time', 'mag', 'mag_err']:
                df[col] = as_numeric(df[col])
            df = df.dropna(subset=['time', 'mag'])
            if len(df) < MIN_LIGHTCURVE_POINTS:
                return pd.DataFrame()
            return df
        except Exception:
            return pd.DataFrame()
    
    def harvest_lightcurves(class_name, candidates, worker, target, clean_so_far, desc, max_workers=None):
        clean_count = clean_so_far
        consecutive_transient = 0
        pauses_used = 0
        pending_verify = 0
    
        def target_reached():
            return clean_count >= target
        for _, outcome in parallel_map(worker, candidates, desc=desc, should_stop=target_reached, max_workers=max_workers):
            if outcome is None:
                outcome = ('rejected', None)
            status, row = outcome
            if status == 'transient':
                consecutive_transient += 1
                if consecutive_transient >= TRANSIENT_ABORT_AFTER:
                    pauses_used += 1
                    if pauses_used > TRANSIENT_MAX_PAUSES:
                        raise NetworkDownError(f'network still failing after {TRANSIENT_MAX_PAUSES} recovery pauses ({pauses_used * TRANSIENT_PAUSE_SECONDS}s) — stopping with progress saved')
                    warn(f'{class_name}: {consecutive_transient} consecutive light-curve failures (server likely throttling); pausing {TRANSIENT_PAUSE_SECONDS}s to recover [pause {pauses_used}/{TRANSIENT_MAX_PAUSES}], then continuing.')
                    time.sleep(TRANSIENT_PAUSE_SECONDS)
                    refresh_gaia_tap_services()
                    consecutive_transient = 0
                continue
            consecutive_transient = 0
            if status == 'pending_verify':
                pending_verify += 1
                continue
            if row is None:
                continue
            checkpoint_append_rows(class_name, [row])
            if status == 'clean':
                clean_count += 1
        if pending_verify:
            print(f'{class_name}: {pending_verify} star(s) received an empty MAST answer this run; they stay PENDING (not rejected) and are re-verified on the next run -- a single empty answer from a busy archive is not proof the star has no data.', flush=True)
        return clean_count
    
    def build_gaia_class(gaia_class):
        star_type = GAIA_CLASS_MAP[gaia_class]
        t_start = time.time()
        print(f'\n================ {star_type} ================')
        clean_already, _ = clean_count_from_checkpoint(star_type)
        if clean_already >= N_PER_CLASS:
            print(f'{star_type}: checkpoint already holds {clean_already} complete rows (target {N_PER_CLASS}); nothing to do.')
            return
        if clean_already:
            print(f'{star_type}: resuming with {clean_already}/{N_PER_CLASS} complete rows from checkpoint.')
        raw_path = RAW_DIR / 'gaia_rrlyr_raw.csv' if star_type == 'RRLYR' else RAW_DIR / f'gaia_{star_type.lower()}_raw.csv'
        top_n = GAIA_TOP_INITIAL
        attempt = 0
        raw = None
        if Path(raw_path).exists():
            try:
                cached = read_csv_ids_safe(raw_path)
                cache_has_required = all((col in cached.columns for col in ['confidence', 'R21', 'R31']))
                if len(cached) >= GAIA_TOP_INITIAL and cache_has_required:
                    raw = cached
                    top_n = len(cached)
                    print(f'{star_type}: reusing cached {raw_path} ({len(cached)} rows).')
                elif not cache_has_required:
                    print(f'{star_type}: cached {raw_path} predates the 9-feature schema; refreshing it.')
            except Exception:
                raw = None
        while True:
            if raw is None or raw.empty:
                raw = query_gaia_pulsator_class(gaia_class, top_n)
                if raw is None or raw.empty:
                    warn(f'{star_type}: Gaia query unavailable (offline?). Progress so far is saved; re-run the cell to retry.')
                    return
            work = raw.copy()
            if 'A1' in work.columns:
                work['A1'] = work['A1'].map(parse_first_float_from_arraylike)
            work['star_type'] = star_type
            work['tic_id'] = np.nan
            work['ogle_id'] = np.nan
            work['label_source'] = 'Gaia_DR3_vari_cepheid_SOS' if gaia_class == 'CEP' else 'Gaia_DR3_vari_classifier_result'
            work['catalogue_used'] = 'Gaia'
            work['feature_source'] = GAIA_SPECIALIZED_TABLES[gaia_class]
            prepared = prepare_common_dataframe(work)
            prepared['quality_flag'] = compute_quality_flags(prepared)
            clean = prepared[strict_clean_mask(prepared)].drop_duplicates(subset=['source_id'])
            print(f'{star_type}: {len(clean)} complete rows among the top {len(prepared)} ranked Gaia candidates.')
            if len(clean) >= N_PER_CLASS:
                break
            attempt += 1
            if attempt >= GAIA_TOP_MAX_ATTEMPTS:
                warn(f'{star_type}: only {len(clean)} complete rows after {attempt} query escalations; keeping what exists.')
                break
            if len(raw) < 0.95 * top_n:
                warn(f'{star_type}: the Gaia catalogue itself is exhausted at {len(raw)} candidates; only {len(clean)} complete rows exist under the current gate.')
                break
            top_n = top_n * GAIA_TOP_ESCALATION
            print(f'{star_type}: not enough complete rows in the top of the ranking; escalating the query to TOP {top_n}.')
            raw = None
        checkpoint_overwrite(star_type, clean)
        print(f'{star_type}: checkpointed {len(clean)} complete rows in {fmt_elapsed(time.time() - t_start)} (mm:ss).')
    
    def make_tess_worker(class_name, cache_prefix, period_col=None, period_range=None):
    
        def process(candidate):
            candidate = dict(candidate)
            try:
                cache_key = f"{cache_prefix}_{candidate.get('source_id')}"
                cached = cache_get_features(cache_key)
                retrying_empty = cached is not None and cached[1] == 'no_data_pending'
                if cached is not None and (not retrying_empty):
                    feats, status = cached
                else:
                    known_p = None
                    if period_col is not None:
                        pv = as_numeric(candidate.get(period_col))
                        if pd.notna(pv) and float(pv) > 0:
                            known_p = float(pv)
                    feats, status = tess_lightcurve_features_fast(tic_id=candidate.get('tic_id'), ra=candidate.get('ra'), dec=candidate.get('dec'), period=known_p, period_range=None if known_p is not None else period_range)
                    got_nothing = True
                    try:
                        got_nothing = not (pd.notna(feats.get('n_points')) and float(feats.get('n_points')) > 0)
                    except Exception:
                        pass
                    if status == 'no_data' and got_nothing and (not retrying_empty):
                        cache_set_features(cache_key, feats, 'no_data_pending')
                        return ('pending_verify', None)
                    cache_set_features(cache_key, feats, status)
                for key, value in feats.items():
                    candidate[key] = value
                if all((pd.notna(candidate.get(k)) for k in ['P', 'amplitude'])):
                    candidate['feature_source'] = 'TESS_lightcurve_fourier'
                row = finalize_candidate_row(candidate)
                return ('clean' if row_is_clean(row) else 'rejected', row)
            except TransientNetworkError:
                return ('transient', None)
            except Exception as exc:
                if _is_transient_exception(exc):
                    return ('transient', None)
                try:
                    return ('rejected', finalize_candidate_row(candidate))
                except Exception:
                    return ('rejected', None)
        return process
    
    def run_tess_lc_class(class_name, ranked, label_source, catalogue_used, confidence_default, cache_prefix, period_col=None, period_range=None):
        t_start = time.time()
        print(f'\n================ {class_name} ================')
        clean_count, processed_keys = clean_count_from_checkpoint(class_name)
        if clean_count >= N_PER_CLASS:
            print(f'{class_name}: checkpoint already holds {clean_count} complete rows (target {N_PER_CLASS}); nothing to do.')
            return
        if clean_count:
            print(f'{class_name}: resuming with {clean_count}/{N_PER_CLASS} complete rows from checkpoint.')
        if ranked is None or ranked.empty or 'source_id' not in ranked.columns:
            warn(f'{class_name}: no ranked candidates available (offline?); will resume on the next run.')
            return
        pending = ranked[~ranked['source_id'].astype(str).isin(processed_keys)].reset_index(drop=True)
        print(f'{class_name}: {len(pending)} ranked candidates left to scan (scanning stops the moment {N_PER_CLASS} complete rows exist).')
        aux_state = {'consec': 0, 'label': class_name}
        cursor = 0
        scanned = 0
        try:
            while clean_count < N_PER_CLASS and cursor < len(pending):
                block = pending.iloc[cursor:cursor + ZHOU_GAIA_BLOCK].copy()
                cursor += len(block)
                scanned += len(block)
                for _c in ['ra', 'dec', 'tic_id']:
                    if _c not in block.columns:
                        block[_c] = np.nan
                aux, skipped = fetch_block_aux_or_skip(block['source_id'].tolist(), aux_state)
                if skipped:
                    continue
                if aux is not None and (not aux.empty):
                    aux = aux.copy()
                    aux['source_id'] = coerce_int64_ids(aux['source_id'])
                    block['_sid_key'] = block['source_id'].astype(str)
                    aux['_sid_key'] = aux['source_id'].astype(str)
                    merged = block.merge(aux.drop(columns=['source_id']), on='_sid_key', how='left')
                    merged = merged.drop(columns=['_sid_key'])
                    if 'gaia_ra' in merged.columns:
                        merged['ra'] = as_numeric(merged['gaia_ra']).combine_first(as_numeric(merged['ra']))
                    if 'gaia_dec' in merged.columns:
                        merged['dec'] = as_numeric(merged['gaia_dec']).combine_first(as_numeric(merged['dec']))
                    merged = merged.drop(columns=['gaia_ra', 'gaia_dec'], errors='ignore')
                else:
                    merged = block
                merged['star_type'] = class_name
                merged['ogle_id'] = np.nan
                merged['label_source'] = label_source
                if 'confidence' in merged.columns:
                    merged['confidence'] = as_numeric(merged['confidence']).fillna(confidence_default)
                else:
                    merged['confidence'] = confidence_default
                merged['catalogue_used'] = catalogue_used
                merged['feature_source'] = 'TESS_lightcurve_fourier'
                tic_backup = merged['tic_id'].copy() if 'tic_id' in merged.columns else pd.Series(np.nan, index=merged.index)
                prepared = prepare_common_dataframe(merged)
                prepared['tic_id'] = tic_backup.map(clean_identifier).values
                gate = gaia_side_gate_mask(prepared)
                rejected = prepared[~gate].copy()
                if not rejected.empty:
                    rejected['quality_flag'] = compute_quality_flags(rejected)
                    checkpoint_append_rows(class_name, [row for _, row in rejected.iterrows()])
                passers = prepared[gate].to_dict('records')
                print(f'{class_name}: block of {len(block)} candidates -> {len(passers)} pass the Gaia gate (others checkpointed as rejected; nothing slow was done for them).')
                if passers:
                    clean_count = harvest_lightcurves(class_name, passers, make_tess_worker(class_name, cache_prefix, period_col, period_range), N_PER_CLASS, clean_count, desc=f'{class_name} TESS light curves ({clean_count}/{N_PER_CLASS} done)')
            if clean_count >= N_PER_CLASS:
                print(f'{class_name}: reached {clean_count}/{N_PER_CLASS} complete rows after scanning {scanned} new candidates in {fmt_elapsed(time.time() - t_start)} (mm:ss).')
            else:
                warn(f'{class_name}: catalogue exhausted with {clean_count}/{N_PER_CLASS} complete rows. All scanned stars are checkpointed; raising CANDIDATES_PER_CLASS will not rescan them.')
        except (NetworkDownError, TransientNetworkError) as exc:
            warn(f'{class_name}: the network appears to be down ({exc}). Progress saved at {clean_count}/{N_PER_CLASS} complete rows — re-run the cell to resume.')
    
    def make_ogle_worker(lc_url_by_id, cache_prefix):
    
        def process(candidate):
            candidate = dict(candidate)
            try:
                ogle_id = str(candidate.get('ogle_id'))
                cache_key = f'{cache_prefix}_{ogle_id}'
                cached = cache_get_features(cache_key)
                if cached is not None:
                    feats, status = cached
                else:
                    lc = read_ogle_lightcurve(lc_url_by_id.get(ogle_id))
                    if lc.empty:
                        feats, status = (feature_dict_nan(n_points=0), 'no_data')
                    else:
                        feats = extract_lightcurve_features(lc['time'].to_numpy(), lc['mag'].to_numpy(), period=as_numeric(candidate.get('P')), mag_err=lc['mag_err'].to_numpy() if 'mag_err' in lc else None)
                        status = 'ok'
                    cache_set_features(cache_key, feats, status)
                for key, value in feats.items():
                    candidate[key] = value
                row = finalize_candidate_row(candidate)
                return ('clean' if row_is_clean(row) else 'rejected', row)
            except TransientNetworkError:
                return ('transient', None)
            except Exception as exc:
                if _is_transient_exception(exc):
                    return ('transient', None)
                try:
                    return ('rejected', finalize_candidate_row(candidate))
                except Exception:
                    return ('rejected', None)
        return process
    
    def run_ogle_lc_class(class_name, catalogue_loader, ranker, label_source, catalogue_used, confidence_default):
        t_start = time.time()
        print(f'\n================ {class_name} ================')
        clean_count, processed_keys = clean_count_from_checkpoint(class_name)
        if clean_count >= N_PER_CLASS:
            print(f'{class_name}: checkpoint already holds {clean_count} complete rows (target {N_PER_CLASS}); nothing to do.')
            return
        if clean_count:
            print(f'{class_name}: resuming with {clean_count}/{N_PER_CLASS} complete rows from checkpoint.')
        catalogue = catalogue_loader()
        if catalogue is None or catalogue.empty:
            return
        ranked = ranker(catalogue)
        cross_cache_path = Path(CHECKPOINT_DIR) / f'cache_{class_name.lower()}_crossmatch.csv'
        pending = ranked[~ranked['ogle_id'].astype(str).isin(processed_keys)].reset_index(drop=True)
        print(f'{class_name}: {len(pending)} ranked OGLE candidates left to scan (bulge first; scanning stops at {N_PER_CLASS} complete rows).')
        aux_state = {'consec': 0, 'label': class_name}
        cursor = 0
        scanned = 0
        try:
            while clean_count < N_PER_CLASS and cursor < len(pending):
                block = pending.iloc[cursor:cursor + EB_CROSSMATCH_BLOCK].copy()
                cursor += len(block)
                scanned += len(block)
                block['ogle_id'] = block['ogle_id'].astype(str)
                matches = ogle_crossmatch_cached(block, cross_cache_path)
                block['source_id'] = block['ogle_id'].map(lambda oid: (matches.get(oid) or {}).get('source_id', np.nan))
                matched_ids = [v for v in block['source_id'] if not pd.isna(v)]
                if matched_ids:
                    aux, skipped = fetch_block_aux_or_skip(matched_ids, aux_state)
                    if skipped:
                        continue
                else:
                    aux = pd.DataFrame(columns=GAIA_AUX_COLUMNS)
                if aux is not None and (not aux.empty):
                    aux = aux.copy()
                    aux['_sid_key'] = aux['source_id'].astype(str).map(lambda s: str(clean_identifier(s)))
                    block['_sid_key'] = block['source_id'].astype(str)
                    merged = block.merge(aux.drop(columns=['source_id']), on='_sid_key', how='left')
                    merged = merged.drop(columns=['_sid_key'])
                    if 'gaia_ra' in merged.columns:
                        merged['ra'] = as_numeric(merged['gaia_ra']).combine_first(as_numeric(merged['ra']))
                    if 'gaia_dec' in merged.columns:
                        merged['dec'] = as_numeric(merged['gaia_dec']).combine_first(as_numeric(merged['dec']))
                    merged = merged.drop(columns=['gaia_ra', 'gaia_dec'], errors='ignore')
                else:
                    merged = block
                merged['star_type'] = class_name
                merged['tic_id'] = np.nan
                merged['label_source'] = label_source
                merged['confidence'] = confidence_default
                merged['catalogue_used'] = catalogue_used
                merged['feature_source'] = 'OGLE_lightcurve_fourier'
                merged['P'] = as_numeric(merged.get('period'))
                lc_url_by_id = dict(zip(merged['ogle_id'].astype(str), merged['lc_file'].astype(str)))
                prepared = prepare_common_dataframe(merged)
                gate = gaia_side_gate_mask(prepared) & as_numeric(prepared['P']).gt(0).fillna(False)
                rejected = prepared[~gate].copy()
                if not rejected.empty:
                    rejected['quality_flag'] = compute_quality_flags(rejected)
                    checkpoint_append_rows(class_name, [row for _, row in rejected.iterrows()])
                passers = prepared[gate].to_dict('records')
                print(f'{class_name}: block of {len(block)} candidates -> {len(passers)} pass the Gaia gate (good parallax + extinction + variability stats).')
                if passers:
                    clean_count = harvest_lightcurves(class_name, passers, make_ogle_worker(lc_url_by_id, class_name), N_PER_CLASS, clean_count, desc=f'{class_name} OGLE light curves ({clean_count}/{N_PER_CLASS} done)', max_workers=OGLE_N_WORKERS)
            if clean_count >= N_PER_CLASS:
                print(f'{class_name}: reached {clean_count}/{N_PER_CLASS} complete rows after scanning {scanned} new candidates in {fmt_elapsed(time.time() - t_start)} (mm:ss).')
            else:
                warn(f'{class_name}: catalogue exhausted with {clean_count}/{N_PER_CLASS} complete rows.')
        except (NetworkDownError, TransientNetworkError) as exc:
            warn(f'{class_name}: the network appears to be down ({exc}). Progress saved at {clean_count}/{N_PER_CLASS} complete rows — re-run the cell to resume.')
    
    def run_eb_class():
        run_ogle_lc_class('EB', load_ogle_eb_catalogue, rank_ogle_eb_candidates, 'OGLE_EB', 'OGLE', 0.95)
    TIC_CACHE_PATH = Path(CHECKPOINT_DIR) / 'cache_tic.csv'
    _tic_cache = None
    
    def _load_tic_cache():
        global _tic_cache
        if _tic_cache is not None:
            return _tic_cache
        _tic_cache = {}
        if TIC_CACHE_PATH.exists():
            try:
                cached = read_csv_ids_safe(TIC_CACHE_PATH)
                for _, row in cached.iterrows():
                    key = clean_identifier(row.get('candidate_key'))
                    tic = clean_identifier(row.get('tic_id'))
                    if not pd.isna(key) and (not pd.isna(tic)):
                        _tic_cache[str(key)] = str(tic)
            except Exception:
                _tic_cache = {}
        return _tic_cache
    
    def get_tic_id_from_radec(ra, dec, radius_arcsec=2):
        try:
            ra = float(ra)
            dec = float(dec)
            if not np.isfinite(ra) or not np.isfinite(dec):
                return np.nan
            coord = SkyCoord(ra * u.deg, dec * u.deg, frame='icrs')
            table = Catalogs.query_region(coord, radius=radius_arcsec * u.arcsec, catalog='TIC')
            if len(table) == 0:
                return np.nan
            colnames = list(table.colnames)
            tic_col = 'ID' if 'ID' in colnames else 'TIC' if 'TIC' in colnames else None
            ra_col = 'ra' if 'ra' in colnames else 'RA' if 'RA' in colnames else None
            dec_col = 'dec' if 'dec' in colnames else 'DEC' if 'DEC' in colnames else None
            if tic_col is None:
                return np.nan
            if ra_col and dec_col:
                coords = SkyCoord(np.asarray(table[ra_col], dtype=float) * u.deg, np.asarray(table[dec_col], dtype=float) * u.deg, frame='icrs')
                idx = int(np.argmin(coord.separation(coords).arcsec))
            else:
                idx = 0
            return clean_identifier(table[tic_col][idx])
        except Exception:
            return np.nan
    
    def fill_missing_tic_ids(df):
        df = df.copy()
        if 'tic_id' not in df.columns:
            df['tic_id'] = np.nan
        cache = _load_tic_cache()
        missing_index = [idx for idx in df.index if pd.isna(clean_identifier(df.at[idx, 'tic_id']))]
        for idx in list(missing_index):
            key = candidate_key_for_row(df.loc[idx])
            if key in cache:
                df.at[idx, 'tic_id'] = cache[key]
                missing_index.remove(idx)
        if not missing_index:
            return df
        new_records = []
    
        def lookup(idx):
            return (idx, get_tic_id_from_radec(df.at[idx, 'ra'], df.at[idx, 'dec'], radius_arcsec=2))
        for _, outcome in parallel_map(lookup, missing_index, desc='MAST TIC cone searches'):
            if outcome is None:
                continue
            idx, tic = outcome
            if not pd.isna(tic):
                df.at[idx, 'tic_id'] = tic
                key = candidate_key_for_row(df.loc[idx])
                cache[key] = str(tic)
                new_records.append({'candidate_key': key, 'tic_id': str(tic)})
        if new_records:
            _ensure_checkpoint_dir()
            with _checkpoint_lock:
                pd.DataFrame(new_records).to_csv(TIC_CACHE_PATH, mode='a', header=not TIC_CACHE_PATH.exists(), index=False)
        still_missing = int(df['tic_id'].map(lambda v: pd.isna(clean_identifier(v))).sum())
        if still_missing:
            warn(f'{still_missing} final rows still have no TIC id (no TIC source within 2"); they are kept because tic_id is a cross-identifier, not one of the ML features.')
        return df
    
    def assemble_final_dataset():
        print('\n================ FINAL ASSEMBLY ================')
        kept_parts = []
        all_parts = []
        for class_name in TARGET_CLASSES:
            ck = load_checkpoint(class_name)
            if ck.empty:
                warn(f'Class {class_name} has 0 processed rows; it will be absent from the final CSV.')
                continue
            all_parts.append(ck.drop(columns=['candidate_key'], errors='ignore'))
            clean = strict_clean_subset(ck).drop_duplicates(subset=['source_id'])
            kept = clean.head(N_PER_CLASS).drop(columns=['candidate_key'], errors='ignore')
            if len(kept) < N_PER_CLASS:
                warn(f'Class {class_name} has only {len(kept)} complete rows so far (target {N_PER_CLASS}). Re-run the cell (it resumes) or check the warnings above.')
            kept_parts.append(kept)
            print(f'{class_name}: {len(kept)} complete rows selected ({len(ck)} candidates processed in total across all runs).')
        before_cleaning = pd.concat(all_parts, ignore_index=True) if all_parts else ensure_final_columns(pd.DataFrame())
        atomic_to_csv(ensure_final_columns(before_cleaning), OUTPUT_DIR / 'combined_before_cleaning.csv')
        print(f'Saved combined_before_cleaning.csv: {len(before_cleaning)} rows (every candidate ever processed).')
        df = pd.concat(kept_parts, ignore_index=True) if kept_parts else ensure_final_columns(pd.DataFrame())
        if not df.empty:
            n_dupes = int(df.duplicated(subset=['source_id']).sum())
            if n_dupes:
                warn(f'Removed {n_dupes} stars that appeared in more than one class (kept first occurrence).')
                df = df.drop_duplicates(subset=['source_id'], keep='first')
            df = fill_missing_tic_ids(df)
            df['quality_flag'] = compute_quality_flags(df)
        df = ensure_final_columns(df)
        atomic_to_csv(df, TRAINING_CSV)
        print('\nRows per class in the final CSV:')
        print(df['star_type'].value_counts(dropna=False))
        if not df.empty:
            required_ok = bool(strict_clean_mask(df).all())
            print(f'Every selected star has required fields present: {required_ok}')
            completeness = df.notna().sum()
            print('Non-null counts per column (final CSV):')
            print(completeness.to_string())
        print('Quality flags:')
        print(df['quality_flag'].value_counts(dropna=False))
        print('Number of TIC IDs found:', int(df['tic_id'].notna().sum()))
        print('Output saved to:', TRAINING_CSV)
        return df
    
    def parse_ogle_lpv_ident_text(text, ra_idx, dec_idx):
        ids, ra_strings, dec_strings = ([], [], [])
        for line in text.splitlines():
            if not line.strip() or line.lstrip().startswith('#'):
                continue
            parts = line.split()
            if len(parts) <= max(ra_idx, dec_idx):
                continue
            ids.append(parts[0])
            ra_strings.append(parts[ra_idx])
            dec_strings.append(parts[dec_idx])
        if not ids:
            return pd.DataFrame()
        try:
            coords = SkyCoord(ra_strings, dec_strings, unit=(u.hourangle, u.deg), frame='icrs')
            return pd.DataFrame({'ogle_id': ids, 'ra': coords.ra.deg, 'dec': coords.dec.deg})
        except Exception:
            rows = []
            for oid, ra_s, dec_s in zip(ids, ra_strings, dec_strings):
                try:
                    coord = SkyCoord(ra_s, dec_s, unit=(u.hourangle, u.deg), frame='icrs')
                    rows.append({'ogle_id': oid, 'ra': float(coord.ra.deg), 'dec': float(coord.dec.deg)})
                except Exception:
                    continue
            return pd.DataFrame(rows)
    
    def parse_ogle_lpv_catalog_text(text):
        rows = []
        for line in text.splitlines():
            if not line.strip() or line.lstrip().startswith('#'):
                continue
            parts = line.split()
            if len(parts) < 5:
                continue
            rows.append({'ogle_id': parts[0], 'I_mag': pd.to_numeric(parts[1], errors='coerce'), 'period': pd.to_numeric(parts[3], errors='coerce'), 'lc_amplitude': pd.to_numeric(parts[4], errors='coerce')})
        return pd.DataFrame(rows)
    
    def load_ogle_lpv_catalogue(class_name):
        cache_path = OGLE_LPV_CATALOG_CACHE[class_name]
        if Path(cache_path).exists():
            try:
                cached = pd.read_csv(cache_path)
                if not cached.empty and 'lc_file' in cached.columns:
                    print(f'OGLE {class_name} catalogue: reusing cached {cache_path} ({len(cached)} rows).')
                    return cached
            except Exception:
                pass
        region, base_url, lc_subdir, catalog_file, ra_idx, dec_idx = OGLE_LPV_SOURCES[class_name]
        try:
            print(f'Downloading OGLE {class_name} catalogue from {base_url} (one-time, cached afterwards)...')
            cat_text = http_get_text_with_retry(f'{base_url}/{catalog_file}', timeout=CATALOG_DOWNLOAD_TIMEOUT)
            ident_text = http_get_text_with_retry(f'{base_url}/ident.dat', timeout=CATALOG_DOWNLOAD_TIMEOUT)
            if cat_text is None or ident_text is None:
                warn(f'OGLE {class_name} catalogue files not found (404); {class_name} skipped.')
                return pd.DataFrame()
            cat_df = parse_ogle_lpv_catalog_text(cat_text)
            ident_df = parse_ogle_lpv_ident_text(ident_text, ra_idx, dec_idx)
        except Exception as exc:
            warn(f'Failed to load OGLE {class_name} catalogue ({exc}); {class_name} resumes on the next run.')
            return pd.DataFrame()
        if cat_df.empty or ident_df.empty:
            warn(f'OGLE {class_name} catalogue parsed empty; {class_name} resumes on the next run.')
            return pd.DataFrame()
        merged = cat_df.merge(ident_df, on='ogle_id', how='inner')
        merged['region'] = region
        merged['lc_file'] = merged['ogle_id'].map(lambda oid: f'{base_url}/{lc_subdir}/{oid}.dat')
        merged = merged.dropna(subset=['ogle_id', 'ra', 'dec', 'period', 'lc_file'])
        merged = merged[as_numeric(merged['period']) > 0]
        atomic_to_csv(merged, cache_path)
        print(f'Saved {cache_path}: {len(merged)} rows')
        return merged
    
    def rank_ogle_lpv_candidates(raw):
        raw = raw.copy()
        for col in ['period', 'I_mag']:
            if col not in raw.columns:
                raw[col] = np.nan
            raw[col] = pd.to_numeric(raw[col], errors='coerce')
        raw = raw.dropna(subset=['ogle_id', 'ra', 'dec', 'lc_file', 'period'])
        raw = raw[raw['period'] > 0]
        raw = raw.drop_duplicates(subset=['ogle_id'], keep='first')
        raw = raw.sort_values(['I_mag', 'ogle_id'], ascending=[True, True], kind='mergesort', na_position='last')
        return raw.reset_index(drop=True)
    
    def run_ogle_lpv_class(class_name):
        run_ogle_lc_class(class_name, lambda: load_ogle_lpv_catalogue(class_name), rank_ogle_lpv_candidates, f'OGLE_{class_name}', 'OGLE', 0.95)
    
    def query_gaia_rotation_ids(top_n):
        fetch_n = int(top_n) * 2
        query = f'\n    SELECT TOP {fetch_n}\n        rm.source_id AS source_id,\n        rm.best_rotation_period AS rot_period,\n        gs.random_index AS random_index,\n        gs.phot_g_mean_mag AS gmag\n    FROM {GAIA_ROTATION_TABLE} AS rm\n    JOIN gaiadr3.gaia_source AS gs ON rm.source_id = gs.source_id\n    WHERE gs.parallax > 0 AND rm.best_rotation_period > 0\n    ORDER BY random_index ASC\n    '
        print(f'ROT: fetching ranked Gaia rotation-modulation source IDs (TOP {fetch_n}, keeping G <= {TESS_HARVEST_MAX_G}).', flush=True)
        print('ROT: this is ONE big archive query with no progress bar -- typically 1-3 minutes. ', flush=True)
        result = gaia_tap_results(query, label=f'ROT ranked-ID query TOP {fetch_n}', prefer='ESA')
        ids = rename_case_insensitive(astropy_to_pandas(result), ['source_id', 'rot_period', 'random_index', 'gmag'])
        if ids.empty:
            return ids
        ids['source_id'] = coerce_int64_ids(ids['source_id'])
        ids['rot_period'] = pd.to_numeric(ids.get('rot_period'), errors='coerce')
        ids['gmag'] = pd.to_numeric(ids.get('gmag'), errors='coerce')
        ids = ids.dropna(subset=['source_id', 'rot_period']).drop_duplicates(subset=['source_id'], keep='first')
        n_fetched = len(ids)
        ids = ids[ids['gmag'].notna() & (ids['gmag'] <= TESS_HARVEST_MAX_G)].reset_index(drop=True)
        print(f'ROT: ranked-ID query returned {n_fetched} candidates; {len(ids)} are bright enough for TESS (G <= {TESS_HARVEST_MAX_G}).', flush=True)
        return ids
    
    def query_gaia_rs_ids(top_n):
        bands = [(-99.0, 12.0), (12.0, 13.0), (13.0, 14.0), (14.0, 15.0), (15.0, float(TESS_HARVEST_MAX_G))]
        print(f'RS: building the candidate list from up to {len(bands)} magnitude-band archive queries, brightest first (~15-60 s each, no progress bar between them). ', flush=True)
        frames = []
        collected = 0
        for lo, hi in bands:
            remaining = int(top_n) - collected
            if remaining <= 0:
                break
            query = f"\n        SELECT TOP {remaining}\n            vc.source_id AS source_id,\n            vc.best_class_score AS confidence,\n            gs.random_index AS random_index,\n            gs.phot_g_mean_mag AS gmag\n        FROM gaiadr3.vari_classifier_result AS vc\n        JOIN gaiadr3.gaia_source AS gs ON vc.source_id = gs.source_id\n        WHERE vc.best_class_name = '{GAIA_RS_CLASS_NAME}'\n            AND vc.best_class_score >= {MIN_CONFIDENCE}\n            AND gs.parallax > 0\n            AND gs.phot_g_mean_mag > {lo} AND gs.phot_g_mean_mag <= {hi}\n        ORDER BY best_class_score DESC, random_index ASC\n        "
            print(f"RS: fetching Gaia 'RS' (RS CVn) source IDs with G in ({lo}, {hi}] (TOP {remaining}).", flush=True)
            result = gaia_tap_results(query, label=f'RS ranked-ID query G ({lo}, {hi}]', prefer='ESA')
            part = rename_case_insensitive(astropy_to_pandas(result), ['source_id', 'confidence', 'random_index', 'gmag'])
            if not part.empty:
                frames.append(part)
                collected += len(part)
        ids = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
        if ids.empty:
            return ids
        ids['source_id'] = coerce_int64_ids(ids['source_id'])
        ids['confidence'] = pd.to_numeric(ids.get('confidence'), errors='coerce')
        ids = ids.dropna(subset=['source_id', 'confidence']).drop_duplicates(subset=['source_id'], keep='first')
        print(f'RS: banded ranked-ID queries returned {len(ids)} candidates, brightest bands first.', flush=True)
        return ids
    
    def build_gaia_rotation_candidates(kind):
        raw_path = GAIA_ROT_RAW_PATH if kind == 'ROT' else GAIA_RS_RAW_PATH
        top_n = max(8000, N_PER_CLASS * 12)
        raw = None
        if Path(raw_path).exists():
            try:
                cached = read_csv_ids_safe(raw_path)
                if not cached.empty and 'source_id' in cached.columns and ('gmag' in cached.columns):
                    raw = cached
                    print(f'{kind}: reusing cached {raw_path} ({len(cached)} candidate ids).')
                elif not cached.empty:
                    print(f'{kind}: cached {raw_path} predates the brightness-aware schema; refreshing it.')
            except Exception:
                raw = None
        if raw is None:
            try:
                raw = query_gaia_rotation_ids(top_n) if kind == 'ROT' else query_gaia_rs_ids(top_n)
            except Exception as exc:
                warn(f'{kind}: Gaia candidate query failed ({exc}); resumes on the next run.')
                return pd.DataFrame()
            if raw is None or raw.empty:
                warn(f'{kind}: Gaia candidate query returned nothing; resumes on the next run.')
                return pd.DataFrame()
            atomic_to_csv(raw, raw_path)
            print(f'Saved {raw_path}: {len(raw)} candidate ids')
        out = pd.DataFrame()
        out['source_id'] = coerce_int64_ids(raw['source_id'])
        out['tic_id'] = np.nan
        out['ra'] = np.nan
        out['dec'] = np.nan
        out['star_type'] = kind
        if kind == 'ROT':
            out['P'] = pd.to_numeric(raw.get('rot_period'), errors='coerce')
            out['confidence'] = 0.95
        else:
            out['P'] = np.nan
            out['confidence'] = pd.to_numeric(raw.get('confidence'), errors='coerce').fillna(MIN_CONFIDENCE)
        out = out.dropna(subset=['source_id']).drop_duplicates(subset=['source_id'], keep='first').reset_index(drop=True)
        return out
    _total_start = time.time()
    print(f'Target: {N_PER_CLASS} complete rows per class (nine features present under the configured quality checks).')
    print('Completed rows are checkpointed. Interruptions may leave requests in progress; rerun to resume.')
    try:
        build_gaia_class('RR')
        build_gaia_class('CEP')
        _need_zhou = any((clean_count_from_checkpoint(c)[0] < N_PER_CLASS for c in ['DSCT', 'GDOR']))
        if _need_zhou:
            zhou_candidates = build_zhou_candidate_table()
            for _class_name in ['DSCT', 'GDOR']:
                if zhou_candidates.empty:
                    warn(f'{_class_name}: no Zhou candidates available (offline?); will resume on the next run.')
                    continue
                _ranked = zhou_candidates[zhou_candidates['star_type'] == _class_name].copy()
                _ranked['_score_sort'] = pd.to_numeric(_ranked['zhou_rank_score'], errors='coerce').fillna(-np.inf)
                _ranked = _ranked.sort_values(['_score_sort', 'zhou_tmag'], ascending=[False, True], kind='mergesort')
                _ranked = _ranked.drop(columns=['_score_sort']).reset_index(drop=True)
                run_tess_lc_class(_class_name, _ranked, 'Zhou_TESS_validated', 'TESS/Zhou', 0.9, 'TESS', period_col=None, period_range=CLASS_PERIOD_RANGES.get(_class_name))
        else:
            print('\nDSCT and GDOR checkpoints already meet the target; Zhou catalogue not needed this run.')
        run_eb_class()
        run_ogle_lpv_class('MIRA')
        run_ogle_lpv_class('SRV')
        for _rot_kind, _rot_conf, _rot_label, _rot_prefix, _rot_pcol in [('ROT', 0.95, 'Gaia_vari_rotation_modulation', 'TESS_ROT', 'P'), ('RS', 0.9, 'Gaia_DR3_vari_classifier_result', 'TESS_RS', None)]:
            if clean_count_from_checkpoint(_rot_kind)[0] >= N_PER_CLASS:
                print(f'\n================ {_rot_kind} ================')
                print(f'{_rot_kind}: checkpoint already holds the target {N_PER_CLASS} complete rows; nothing to do.')
                continue
            _rot_ranked = build_gaia_rotation_candidates(_rot_kind)
            if _rot_ranked.empty:
                warn(f'{_rot_kind}: no candidates available (offline?); will resume on the next run.')
                continue
            run_tess_lc_class(_rot_kind, _rot_ranked, _rot_label, 'Gaia/TESS', _rot_conf, _rot_prefix, period_col=_rot_pcol, period_range=CLASS_PERIOD_RANGES[_rot_kind])
        final_df = assemble_final_dataset()
        print(f'\nTotal wall time this run: {fmt_elapsed(time.time() - _total_start)} (mm:ss).')
        if _WARNINGS:
            print(f'\n{len(_WARNINGS)} warning(s) were printed above — scroll up for details.')
        print('Reminder: raising CANDIDATES_PER_CLASS and re-running only does the NEW work; use an empty cache directory for a fresh rebuild.')
        print(final_df.head())
        print(final_df.columns.tolist())
    except KeyboardInterrupt:
        print('\n--------------------------------------------------------------')
        print('Interrupted. All completed stars are saved in checkpoints/ and feature_cache/.')
        print('Just run this cell again whenever you like: it resumes where it stopped.')
        print('--------------------------------------------------------------')
        raise
else:
    print("Stage 1 skipped; set RUN_BUILD_TRAINING = True to harvest archive data.")


## Stage 2: train and evaluate

Fit the random forest to the frozen training table. The holdout report and five-fold out-of-fold predictions measure agreement with the harvested reference labels. They do not establish accuracy on a different survey or detect all unrepresented variable types. This stage writes the model, reports and training-data checksum.


In [ ]:
def train_classifier():
    require_packages({"pandas":"pandas", "numpy":"numpy", "sklearn":"scikit-learn", "joblib":"joblib"})
    from pathlib import Path
    from datetime import datetime, timezone
    import pandas as pd
    import numpy as np
    import joblib
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score, classification_report
    from sklearn.model_selection import train_test_split, StratifiedKFold
    training_path = Path(TRAINING_CSV)
    if not training_path.exists():
        raise FileNotFoundError(f'Training CSV not found: {TRAINING_CSV}. Provide the training CSV or run stage 1.')
    train_df = read_csv_with_ids(training_path)
    required_columns = ML_FEATURES + ['star_type']
    missing_columns = [col for col in required_columns if col not in train_df.columns]
    if missing_columns:
        raise ValueError(f'Training CSV is missing required columns: {missing_columns}')
    train_df = train_df.copy()
    for col in ML_FEATURES:
        train_df[col] = pd.to_numeric(train_df[col], errors='coerce')
    if 'confidence' in train_df.columns:
        train_df['confidence'] = pd.to_numeric(train_df['confidence'], errors='coerce')
    else:
        train_df['confidence'] = 1.0
    clean_mask = np.isfinite(train_df[ML_FEATURES]).all(axis=1) & train_df['star_type'].notna()
    if 'quality_flag' in train_df.columns:
        clean_mask &= train_df['quality_flag'].eq('good')
    train_df = train_df.loc[clean_mask].reset_index(drop=True)
    if train_df.empty:
        raise ValueError('No complete clean training rows were found. Rebuild the training CSV first.')
    if train_df['star_type'].nunique() < 2:
        raise ValueError('At least two star classes are required to train a classifier.')
    X = train_df[ML_FEATURES]
    y = train_df['star_type'].astype(str)
    sample_weight = train_df['confidence'].fillna(1.0)
    if not np.isfinite(sample_weight).all() or not sample_weight.between(0.0, 1.0).all():
        raise ValueError("Training confidence weights must be finite and between 0 and 1.")
    if y.value_counts().min() < 5:
        raise ValueError("Five-fold cross-validation requires at least five complete rows per class.")
    if "source_id" in train_df.columns and train_df["source_id"].dropna().duplicated().any():
        raise ValueError("Duplicate source_id values would invalidate the train/test split; deduplicate first.")
    if sample_weight.sum() <= 0 or sample_weight.groupby(y).sum().le(0).any():
        raise ValueError("Each training class must have positive total sample weight.")
    for output_path in (MODEL_PATH, TRAINING_REPORT_CSV, TRUTH_TABLE_CSV):
        prepare_output_path(output_path)
    print('Training rows per class:')
    print(y.value_counts())
    missing_target_classes = [class_name for class_name in TARGET_CLASSES if class_name not in set(y)]
    if missing_target_classes and REQUIRE_ALL_TARGET_CLASSES_FOR_TRAINING:
        raise ValueError(f'Training CSV does not contain all target classes. Missing: {missing_target_classes}. Re-run the dataset-builder cell until the training CSV has all classes you want the model to learn.')
    elif missing_target_classes:
        print('WARNING: training without these target classes:', missing_target_classes)
    model = RandomForestClassifier(n_estimators=500, max_features='sqrt', min_samples_leaf=2, class_weight='balanced_subsample', n_jobs=-1, random_state=RANDOM_STATE)
    can_holdout = len(train_df) >= 20 and y.value_counts().min() >= 2
    report_rows = []
    if can_holdout:
        X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(X, y, sample_weight, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
        holdout_model = RandomForestClassifier(n_estimators=500, max_features='sqrt', min_samples_leaf=2, class_weight='balanced_subsample', n_jobs=-1, random_state=RANDOM_STATE)
        holdout_model.fit(X_train, y_train, sample_weight=w_train)
        pred = holdout_model.predict(X_test)
        acc = accuracy_score(y_test, pred)
        print(f'Holdout accuracy: {acc:.4f}')
        print(classification_report(y_test, pred, zero_division=0))
        report = classification_report(y_test, pred, zero_division=0, output_dict=True)
        for label, metrics in report.items():
            if isinstance(metrics, dict):
                row = {'label': label}
                row.update(metrics)
                report_rows.append(row)
    else:
        print('Holdout report skipped because there are too few rows or too few rows per class.')
    model.fit(X, y, sample_weight=sample_weight)
    feature_importances = dict(sorted(zip(ML_FEATURES, (float(v) for v in model.feature_importances_)), key=lambda kv: kv[1], reverse=True))
    print('\nFeature importances (most -> least):')
    for fname, importance in feature_importances.items():
        print(f'  {fname:12s} {importance:.3f}')
    Xv = X.to_numpy()
    yv = y.to_numpy()
    wv = sample_weight.to_numpy()
    cv_classes = np.array(sorted(pd.unique(yv)))
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    oof_proba = np.zeros((len(yv), len(cv_classes)))
    for tr_idx, te_idx in skf.split(Xv, yv):
        fold_model = RandomForestClassifier(n_estimators=500, max_features='sqrt', min_samples_leaf=2, class_weight='balanced_subsample', n_jobs=-1, random_state=RANDOM_STATE)
        fold_model.fit(Xv[tr_idx], yv[tr_idx], sample_weight=wv[tr_idx])
        col = [list(fold_model.classes_).index(c) for c in cv_classes]
        oof_proba[te_idx] = fold_model.predict_proba(Xv[te_idx])[:, col]
    cv_pred = cv_classes[np.argmax(oof_proba, axis=1)]
    cv_conf = oof_proba[np.arange(len(yv)), np.argmax(oof_proba, axis=1)]
    cv_recognised = cv_conf >= MIN_CONFIDENCE_TO_LABEL
    cv_label = np.where(cv_recognised, cv_pred, 'UNKNOWN')
    cv_correct = cv_pred == yv
    truth_table = pd.DataFrame({'source_id': train_df.get('source_id', pd.Series(np.arange(len(yv)))).values, 'star_type': yv, 'catalog_confidence': np.round(wv, 3), 'cv_predicted_type': cv_pred, 'cv_confidence': np.round(cv_conf, 3), 'cv_label': cv_label, 'cv_correct': cv_correct})
    for ci, cname in enumerate(cv_classes):
        truth_table[f'cv_prob_{cname}'] = np.round(oof_proba[:, ci], 3)
    truth_table.to_csv(TRUTH_TABLE_CSV, index=False)
    print('\n=== Training truth table (five-fold out-of-fold predictions) ===')
    print(f'Overall CV accuracy: {cv_correct.mean():.3f}')
    print('Per-class: n, accuracy, confidence min/median/mean')
    for c in cv_classes:
        sel = yv == c
        cc = cv_conf[sel]
        print(f'  {c:5s} n={int(sel.sum()):3d} acc={cv_correct[sel].mean():.3f} | conf {cc.min():.2f}/{np.median(cc):.2f}/{cc.mean():.2f}')
    print('Confusion matrix (rows=true, cols=cv_pred):')
    print(pd.crosstab(pd.Series(yv, name='true'), pd.Series(cv_pred, name='pred')).to_string())
    print(f"\nConfidence threshold at MIN_CONFIDENCE_TO_LABEL={MIN_CONFIDENCE_TO_LABEL}: {int((~cv_recognised).sum())}/{len(yv)} training stars flagged ({int((~cv_recognised & cv_correct).sum())} of them were actually correct).")
    print('Threshold sweep (flagged / correct-but-flagged):')
    for thr in [0.35, 0.4, 0.45, 0.5, 0.55, 0.6]:
        fl = cv_conf < thr
        print(f'  thr={thr:.2f}: {int(fl.sum()):3d}/{len(yv)} flagged ({int((fl & cv_correct).sum())} correct-but-flagged)')
    print(f'Truth table saved to: {TRUTH_TABLE_CSV}')
    from importlib.metadata import version
    bundle = {"training_csv_sha256": sha256_file(training_path),
              "package_versions": {name: version(name) for name in ["pandas", "numpy", "scikit-learn", "joblib"]},
              "feature_definition_version": "legacy-nine-feature-v1",'model': model, 'features': ML_FEATURES, 'classes': list(model.classes_), 'target_classes': TARGET_CLASSES, 'training_csv': TRAINING_CSV, 'n_training_rows': int(len(train_df)), 'rows_per_class': y.value_counts().to_dict(), 'feature_importances': feature_importances, 'min_confidence_to_label': float(MIN_CONFIDENCE_TO_LABEL), 'cv_accuracy': float(cv_correct.mean()), 'created_utc': datetime.now(timezone.utc).isoformat(timespec='seconds')}
    joblib.dump(bundle, MODEL_PATH)
    if report_rows:
        pd.DataFrame(report_rows).to_csv(TRAINING_REPORT_CSV, index=False)
        print('Training report saved to:', TRAINING_REPORT_CSV)
    print('Model saved to:', MODEL_PATH)
    print('Classes:', list(model.classes_))
    print(f'Prediction cell will tag UNKNOWN below confidence {MIN_CONFIDENCE_TO_LABEL}.')
    return bundle

if RUN_TRAIN_MODEL:
    trained_bundle = train_classifier()
else:
    print("Stage skipped; set RUN_TRAIN_MODEL = True in the setup cell to run it.")


## Stage 3: classify a prepared feature table

Supply PREDICTION_INPUT_CSV with the nine named features, using the conventions documented in README.md. The notebook does not derive these features from a raw light curve. Only load a model file you trust. UNKNOWN means that the highest class probability is below the configured threshold; it is not an independent validation of an unfamiliar class.


In [ ]:
def predict_classes():
    require_packages({"pandas":"pandas", "numpy":"numpy", "sklearn":"scikit-learn", "joblib":"joblib"})
    from pathlib import Path
    import pandas as pd
    import numpy as np
    import joblib
    model_path = Path(MODEL_PATH)
    if not model_path.exists():
        raise FileNotFoundError(f'Saved model not found: {MODEL_PATH}. Provide the model file or run stage 2.')
    input_path = Path(PREDICTION_INPUT_CSV)
    if not input_path.exists():
        raise FileNotFoundError(f'Prediction input CSV not found: {PREDICTION_INPUT_CSV}. Set PREDICTION_INPUT_CSV to a CSV containing the nine documented features.')
    bundle = joblib.load(model_path)
    model = bundle['model']
    if list(bundle['features']) != ML_FEATURES:
        raise ValueError(f"Saved model uses {bundle['features']}, but this prediction cell requires {ML_FEATURES}. Use a model trained with the nine documented features.")
    confidence_threshold = PREDICTION_CONFIDENCE_OVERRIDE if PREDICTION_CONFIDENCE_OVERRIDE is not None else float(bundle.get('min_confidence_to_label', 0.45))
    df_new = read_csv_with_ids(input_path)
    missing_features = [col for col in ML_FEATURES if col not in df_new.columns]
    if missing_features:
        raise ValueError(f'Prediction input CSV is missing required ML feature columns: {missing_features}')
    features = df_new[ML_FEATURES].copy()
    for col in ML_FEATURES:
        features[col] = pd.to_numeric(features[col], errors='coerce')
    incomplete_mask = ~np.isfinite(features).all(axis=1)
    if incomplete_mask.any():
        bad_count = int(incomplete_mask.sum())
        raise ValueError(f'Prediction input has {bad_count} rows with missing, non-numeric or infinite ML features. Fix or remove those rows before prediction; the classifier will not invent features.')
    
    def choose_star_id(row):
        for col in ['star_id', 'source_id', 'gaia_id']:
            if col in row.index and pd.notna(row[col]) and (str(row[col]).strip().lower() not in {'', 'nan', 'none', 'masked'}):
                value = row[col]
                if isinstance(value, (int, np.integer)):
                    return int(value)
                if isinstance(value, (float, np.floating)) and np.isfinite(value) and float(value).is_integer():
                    return int(value)
                return value
        return f'row_{row.name}'
    if 'source_id' in df_new.columns:
        gaia_id = df_new['source_id']
    elif 'gaia_id' in df_new.columns:
        gaia_id = df_new['gaia_id']
    else:
        gaia_id = np.nan
    if df_new.empty:
        raise ValueError("Prediction input has no rows.")
    if not np.isfinite(confidence_threshold) or not 0.0 <= confidence_threshold <= 1.0:
        raise ValueError("The prediction confidence threshold must lie between 0 and 1.")
    proba = model.predict_proba(features)
    best_idx = np.argmax(proba, axis=1)
    best_class = np.asarray(model.classes_)[best_idx]
    confidence_score = proba[np.arange(len(df_new)), best_idx]
    recognised = confidence_score >= confidence_threshold
    star_type = np.where(recognised, best_class, "UNKNOWN")
    out = pd.DataFrame({'star_id': df_new.apply(choose_star_id, axis=1), 'gaia_id': gaia_id, 'star_type': star_type, 'confidence_score': confidence_score, 'recognised': recognised, 'best_guess_type': best_class})
    for class_index, class_name in enumerate(model.classes_):
        out[f'prob_{class_name}'] = proba[:, class_index]
    out.to_csv(prepare_output_path(PREDICTION_OUTPUT_CSV), index=False)
    print('Predictions saved to:', PREDICTION_OUTPUT_CSV)
    print(f'Predicted rows: {len(out)} | confidence threshold for a confident label: {confidence_threshold}')
    n_unknown = int((~recognised).sum())
    print(f"Flagged 'below threshold' (UNKNOWN): {n_unknown}/{len(out)}")
    print('Assigned classes (UNKNOWN = below confidence threshold):')
    print(out['star_type'].value_counts(dropna=False))
    print(out.head())
    return out

if RUN_PREDICT:
    predictions = predict_classes()
else:
    print("Stage skipped; set RUN_PREDICT = True in the setup cell to run it.")
